In [1]:
# ============================================================
# FULL PIPELINE (State included + Emotion analysis + More plots)
# Sustainable policy attitudes paper:
#   City council discourse (caption_text_clean) vs Amazon reviews (text)
#
# Inputs:
#   Amazon: "Data analysis\filtered_reviews_with_meta.xlsx"   (uses column: text)
#   City:   "Data analysis/combined_data_2006_2023.xlsx"      (uses column: caption_text_clean)
#
# Key features:
# - Shared keyword dictionary across both corpora
# - Extract +/- 400 char windows around keyword hits
# - Merge overlapping windows + keep multi-keyword hits
# - Theme tagging from keyword categories
# - Sentiment (VADER if available; fallback neutral)
# - Stance (support / oppose / conditional / neutral) via rule-based markers
# - Emotion analysis:
#     * Tries NRCLex if installed
#     * Else uses a lightweight built-in emotion lexicon fallback
# - Adds STATE name into final outputs (robust column detection + abbrev->full mapping)
# - More plotting:
#     * stance distribution (by corpus)
#     * top themes (by corpus)
#     * sentiment-by-theme heatmap
#     * emotion profile by corpus
#     * dominant emotion distribution
#     * city stance by state (top states)
#     * city stance trend by year (if year/date available)
#     * amazon sentiment vs rating
#
# Outputs (in ./outputs):
#   extracted_windows.xlsx
#   summary_tables.xlsx
#   figures/*.png
# ============================================================

import re
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Config
# -----------------------------
WINDOW_CHARS = 400
MIN_TEXT_LEN = 20

AMAZON_PATH = Path(r"Data analysis\filtered_reviews_with_meta.xlsx")
CITY_PATH   = Path(r"Data analysis\combined_data_2006_2023.xlsx")

# Candidate names for text cols
AMAZON_TEXT_COL_CANDIDATES = ["text", "Text", "review_text", "reviewText"]
CITY_TEXT_COL_CANDIDATES   = ["caption_text_clean", "Caption_Text_Clean", "caption_text", "Caption_Text"]

# Candidate names for state cols (city file)
CITY_STATE_COL_CANDIDATES  = ["state", "state_name", "state_abbrev", "state_code", "st", "us_state"]

# Candidate names for date/year cols (city file)
CITY_YEAR_COL_CANDIDATES   = ["year", "Year"]
CITY_DATE_COL_CANDIDATES   = ["date", "Date", "meeting_date", "Meeting_Date", "timestamp", "Timestamp"]

OUTPUT_DIR = Path("outputs")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Optional: speed testing on smaller subset
MAX_ROWS_AMAZON = None  # e.g., 50000
MAX_ROWS_CITY   = None  # e.g., 50000

# ============================================================
# Utilities
# ============================================================
def normalize_text(s) -> str:
    if not isinstance(s, str):
        return ""
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def canonicalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    # lowercase if safe
    lower = [c.lower() for c in df.columns]
    if len(set(lower)) == len(lower):
        df.columns = lower
    return df

def find_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols = list(df.columns)
    for c in candidates:
        if c in cols:
            return c
    lower_map = {c.lower(): c for c in cols}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

def load_excel(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")
    return pd.read_excel(path)

# ============================================================
# US State abbreviation -> full name mapping
# ============================================================
US_STATE_ABBREV_TO_NAME = {
    "AL":"Alabama","AK":"Alaska","AZ":"Arizona","AR":"Arkansas","CA":"California","CO":"Colorado","CT":"Connecticut",
    "DE":"Delaware","FL":"Florida","GA":"Georgia","HI":"Hawaii","ID":"Idaho","IL":"Illinois","IN":"Indiana",
    "IA":"Iowa","KS":"Kansas","KY":"Kentucky","LA":"Louisiana","ME":"Maine","MD":"Maryland","MA":"Massachusetts",
    "MI":"Michigan","MN":"Minnesota","MS":"Mississippi","MO":"Missouri","MT":"Montana","NE":"Nebraska","NV":"Nevada",
    "NH":"New Hampshire","NJ":"New Jersey","NM":"New Mexico","NY":"New York","NC":"North Carolina","ND":"North Dakota",
    "OH":"Ohio","OK":"Oklahoma","OR":"Oregon","PA":"Pennsylvania","RI":"Rhode Island","SC":"South Carolina",
    "SD":"South Dakota","TN":"Tennessee","TX":"Texas","UT":"Utah","VT":"Vermont","VA":"Virginia","WA":"Washington",
    "WV":"West Virginia","WI":"Wisconsin","WY":"Wyoming","DC":"District of Columbia"
}

def normalize_state_value(x) -> Tuple[Optional[str], Optional[str]]:
    """
    Returns (state_abbrev, state_name) if possible.
    Accepts "CA", "California", "california", etc.
    """
    if not isinstance(x, str):
        return (None, None)
    s = x.strip()
    if not s:
        return (None, None)

    # If looks like abbreviation
    if len(s) == 2 and s.upper() in US_STATE_ABBREV_TO_NAME:
        ab = s.upper()
        return (ab, US_STATE_ABBREV_TO_NAME[ab])

    # Try match full name (case-insensitive)
    s_low = s.lower()
    for ab, name in US_STATE_ABBREV_TO_NAME.items():
        if s_low == name.lower():
            return (ab, name)

    # Sometimes stored as "CA - California"
    m = re.match(r"^\s*([A-Za-z]{2})\s*[-,]\s*(.+)\s*$", s)
    if m:
        ab = m.group(1).upper()
        if ab in US_STATE_ABBREV_TO_NAME:
            return (ab, US_STATE_ABBREV_TO_NAME[ab])

    # Could be something else (keep as name)
    # Try title-case as a "name"
    return (None, s.title())

def add_city_state_columns(city_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds:
      - state_raw (original)
      - state_abbrev
      - state_name (full)
    Uses robust column detection.
    """
    df = city_df.copy()
    state_col = find_column(df, CITY_STATE_COL_CANDIDATES)
    if state_col is None:
        # Ensure columns exist even if missing
        df["state_raw"] = np.nan
        df["state_abbrev"] = np.nan
        df["state_name"] = np.nan
        return df

    df["state_raw"] = df[state_col].astype(str)

    ab_list = []
    nm_list = []
    for v in df["state_raw"].tolist():
        ab, nm = normalize_state_value(v)
        ab_list.append(ab)
        nm_list.append(nm)
    df["state_abbrev"] = ab_list
    df["state_name"] = nm_list
    return df

def add_city_year_column(city_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds 'year' if missing using date column (if available).
    """
    df = city_df.copy()

    year_col = find_column(df, CITY_YEAR_COL_CANDIDATES)
    if year_col is not None and "year" not in df.columns:
        df["year"] = pd.to_numeric(df[year_col], errors="coerce")

    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        return df

    date_col = find_column(df, CITY_DATE_COL_CANDIDATES)
    if date_col is not None:
        # Try parse to datetime
        dt = pd.to_datetime(df[date_col], errors="coerce")
        df["year"] = dt.dt.year
    else:
        df["year"] = np.nan
    return df

# ============================================================
# Keyword dictionary (shared)
# ============================================================
KEYWORDS_BY_CATEGORY: Dict[str, List[str]] = {
    "policy_regulation": [
        "policy", "policies", "regulation", "regulations", "regulatory",
        "law", "laws", "ordinance", "ordinances", "code", "compliance",
        "mandate", "mandated", "requirement", "required", "rule", "rules",
        "standard", "standards", "enforcement", "enforce", "penalty", "penalties",
        "fine", "fines", "citation", "permit", "permitting",
        "initiative", "program", "plan", "strategy", "roadmap",
        "target", "targets", "goal", "goals", "benchmark", "benchmarks",
        "public comment", "public hearing", "stakeholder", "community input"
    ],
    "sustainability_environment": [
        "sustainability", "sustainable",
        "environment", "environmental",
        "climate", "climate action", "climate-friendly",
        "carbon", "carbon footprint", "emissions", "greenhouse gas", "ghg",
        "energy efficient", "energy efficiency",
        "pollution", "conservation", "ecological",
        "eco-friendly", "green product", "green"
    ],
    "circular_design": [
        "circular economy", "circular",
        "lifecycle", "life cycle", "product lifecycle",
        "durability", "durable", "long-lasting", "longevity", "lifespan",
        "repair", "repairable", "repairability", "fixable", "serviceable",
        "right to repair", "repair rights",
        "modular", "modularity",
        "reusable", "reuse", "refill", "refillable",
        "recyclable", "recyclability", "recycle", "recycling",
        "compostable", "compost", "biodegradable",
        "recycled content", "post-consumer recycled", "pcr",
        "eco design", "ecodesign", "design for environment", "dfe",
        "product stewardship", "stewardship",
        "take-back", "takeback", "return-to-manufacturer"
    ],
    "packaging_single_use": [
        "packaging", "packaged", "packaging waste",
        "plastic", "plastics", "single-use", "single use", "disposable",
        "overpackaging", "excess packaging",
        "recyclable packaging", "compostable packaging",
        "paper packaging", "cardboard",
        "bubble wrap", "air pillows", "plastic wrap",
        "bag ban", "plastic bag ban",
        "foam ban", "styrofoam", "polystyrene",
        "container deposit", "bottle bill",
        "refill station", "bulk buying", "bulk refill"
    ],
    "ecommerce_logistics": [
        "e-commerce", "ecommerce",
        "online shopping", "online purchase", "internet retail", "digital marketplace",
        "delivery", "shipping", "shipped",
        "last-mile delivery", "last mile",
        "delivery emissions", "shipping emissions",
        "returns", "return policy", "reverse logistics", "refund", "refunded",
        "warehouse", "fulfillment center",
        "subscription delivery", "recurring delivery"
    ],
    "ewaste_electronics": [
        "electronics", "electronic",
        "e-waste", "ewaste", "electronic waste",
        "device disposal", "dispose of electronics",
        "recycling center", "drop-off", "drop off", "collection event",
        "refurbish", "refurbished", "refurbishment",
        "repair shop", "repair cafe", "repair café",
        "reuse electronics", "device reuse",
        "planned obsolescence", "obsolescence",
        "hazardous waste", "toxic", "heavy metals",
        "lithium battery", "battery recycling", "battery disposal"
    ],
    "labels_trust_transparency": [
        "eco-label", "ecolabel", "eco label",
        "certification", "certified", "third-party certified", "third party certified",
        "epeat", "energy star", "fsc", "fair trade",
        "label claim", "sustainability claim",
        "transparency", "traceability",
        "greenwashing", "misleading", "false claims", "fake", "scam",
        "verified", "verification", "accountability"
    ],
    "consumer_tradeoffs": [
        "affordability", "affordable",
        "expensive", "cost", "costs", "cost increase", "higher price",
        "value", "value for money", "worth it",
        "quality", "low quality", "cheap", "durable quality",
        "convenience", "inconvenient", "hassle", "effort",
        "availability", "hard to find",
        "performance", "functionality", "reliability"
    ],
    "waste_systems": [
        "waste management", "solid waste",
        "landfill", "diversion", "waste diversion",
        "composting program", "organics program",
        "recycling program", "mrf",
        "contamination",
        "collection", "pickup", "curbside",
        "bin", "cart", "trash cart",
        "sanitation department"
    ],
}

# ============================================================
# Stance markers
# ============================================================
STANCE_MARKERS = {
    "support": [
        "support", "supported", "in favor", "favor", "favour", "approve", "approved",
        "agree", "aligned", "beneficial", "necessary", "important", "critical",
        "good idea", "great", "positive", "progress", "solution",
        "protect", "reduce waste", "reduce emissions"
    ],
    "oppose": [
        "oppose", "opposed", "against", "reject", "rejected",
        "don't support", "do not support", "no support",
        "waste of money", "pointless", "too expensive", "cost burden",
        "government overreach", "unfair", "unreasonable",
        "doesn't work", "does not work", "ineffective",
        "skeptical", "sceptical", "doubt", "concern", "concerns"
    ],
    "conditional": [
        "support but", "agree but", "as long as", "provided that",
        "depends", "only if", "if we can",
        "need funding", "needs funding", "needs enforcement",
        "implementation challenges", "feasibility", "equity", "small businesses",
        "tradeoff", "trade-off", "tradeoffs"
    ]
}

# ============================================================
# Keyword regex build
# ============================================================
def build_keyword_regex(keywords_by_cat: Dict[str, List[str]]) -> Tuple[re.Pattern, Dict[str, str]]:
    term_to_cat = {}
    all_terms = []
    for cat, terms in keywords_by_cat.items():
        for t in terms:
            tl = t.lower().strip()
            term_to_cat[tl] = cat
            all_terms.append(tl)

    all_terms = sorted(set(all_terms), key=len, reverse=True)
    escaped = [re.escape(t) for t in all_terms]
    pattern = r"(?<!\w)(" + "|".join(escaped) + r")(?!\w)"
    compiled = re.compile(pattern, flags=re.IGNORECASE)
    return compiled, term_to_cat

KEYWORD_REGEX, TERM_TO_CAT = build_keyword_regex(KEYWORDS_BY_CATEGORY)

# ============================================================
# Window extraction + merge
# ============================================================
def extract_windows_for_text(text: str, window_chars: int) -> List[Tuple[int, int, List[str]]]:
    text = normalize_text(text)
    if len(text) < MIN_TEXT_LEN:
        return []
    hits = []
    for m in KEYWORD_REGEX.finditer(text):
        term = m.group(1).lower()
        start = max(0, m.start() - window_chars)
        end = min(len(text), m.end() + window_chars)
        hits.append((start, end, term))
    if not hits:
        return []
    return [(s, e, [t]) for s, e, t in hits]

def merge_overlapping_windows(windows: List[Tuple[int, int, List[str]]], gap: int = 0) -> List[Tuple[int, int, List[str]]]:
    if not windows:
        return []
    windows_sorted = sorted(windows, key=lambda x: (x[0], x[1]))
    merged = []
    cur_s, cur_e, cur_terms = windows_sorted[0][0], windows_sorted[0][1], list(windows_sorted[0][2])
    for s, e, terms in windows_sorted[1:]:
        if s <= cur_e + gap:
            cur_e = max(cur_e, e)
            cur_terms.extend(terms)
        else:
            merged.append((cur_s, cur_e, sorted(set(cur_terms))))
            cur_s, cur_e, cur_terms = s, e, list(terms)
    merged.append((cur_s, cur_e, sorted(set(cur_terms))))
    return merged

def terms_to_categories(terms: List[str]) -> List[str]:
    cats = []
    for t in terms:
        cat = TERM_TO_CAT.get(t.lower())
        if cat:
            cats.append(cat)
    return sorted(set(cats))

# ============================================================
# Sentiment (VADER) with fallback
# ============================================================
def init_sentiment_analyzer():
    # NLTK VADER
    try:
        from nltk.sentiment import SentimentIntensityAnalyzer
        try:
            return SentimentIntensityAnalyzer()
        except Exception:
            try:
                import nltk
                nltk.download("vader_lexicon")
                return SentimentIntensityAnalyzer()
            except Exception:
                pass
    except Exception:
        pass

    # vaderSentiment package
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as VSIA
        return VSIA()
    except Exception:
        return None

SENT_ANALYZER = init_sentiment_analyzer()

def vader_sentiment(text: str) -> Tuple[float, str]:
    s = normalize_text(text)
    if not s or SENT_ANALYZER is None:
        return 0.0, "neutral"
    try:
        score = SENT_ANALYZER.polarity_scores(s)["compound"]
    except Exception:
        return 0.0, "neutral"
    if score >= 0.05:
        return score, "positive"
    if score <= -0.05:
        return score, "negative"
    return score, "neutral"

# ============================================================
# Stance classification
# ============================================================
def _count_markers(text: str, markers: List[str]) -> int:
    t = text.lower()
    c = 0
    for mk in markers:
        pat = re.compile(r"(?<!\w)" + re.escape(mk.lower()) + r"(?!\w)")
        c += len(pat.findall(t))
    return c

def classify_stance(text: str) -> str:
    t = normalize_text(text)
    if not t:
        return "neutral"
    sup = _count_markers(t, STANCE_MARKERS["support"])
    opp = _count_markers(t, STANCE_MARKERS["oppose"])
    cond = _count_markers(t, STANCE_MARKERS["conditional"])

    if cond > 0 and (sup > 0 or opp > 0):
        return "conditional"
    if sup > 0 and opp > 0:
        return "conditional"
    if sup > opp and sup > 0:
        return "support"
    if opp > sup and opp > 0:
        return "oppose"
    return "neutral"

# ============================================================
# Emotion analysis
# 1) Try NRCLex (if installed)
# 2) Else fallback lexicon (small but functional)
# ============================================================
EMOTION_LABELS = ["anger", "fear", "joy", "sadness", "disgust", "surprise", "trust", "anticipation"]

def try_init_nrclex():
    try:
        from nrclex import NRCLex
        return NRCLex
    except Exception:
        return None

NRCLex = try_init_nrclex()

# Small fallback lexicon (expand anytime)
FALLBACK_EMO_LEX = {
    "anger": {"angry","anger","furious","outrage","annoyed","mad","irritated","frustrated","hate","hated","rage"},
    "fear": {"fear","afraid","scared","worry","worried","anxious","risk","danger","unsafe","threat"},
    "joy": {"happy","joy","glad","pleased","love","loved","excellent","great","amazing","wonderful","delight"},
    "sadness": {"sad","sadness","disappointed","regret","unhappy","depressed","down","sorry","upset"},
    "disgust": {"disgust","disgusting","gross","nasty","filthy","dirty"},
    "surprise": {"surprise","surprised","shocked","unexpected","wow"},
    "trust": {"trust","trusted","reliable","credible","legit","verified","authentic","honest","confidence"},
    "anticipation": {"anticipate","anticipation","hope","hopeful","expect","expected","looking forward","plan"}
}

TOKEN_RE = re.compile(r"[A-Za-z']+")

def emotion_scores(text: str) -> Dict[str, float]:
    """
    Returns normalized emotion scores (0..1) across EMOTION_LABELS.
    """
    t = normalize_text(text)
    if not t:
        return {e: 0.0 for e in EMOTION_LABELS}

    # NRCLex if available
    if NRCLex is not None:
        try:
            emo = NRCLex(t)
            # NRCLex returns raw frequencies in affect_frequencies (normalized by word count)
            freqs = emo.affect_frequencies
            out = {e: float(freqs.get(e, 0.0)) for e in EMOTION_LABELS}
            return out
        except Exception:
            pass

    # Fallback: count words in small lexicon, normalize by token count
    tokens = [w.lower() for w in TOKEN_RE.findall(t)]
    if not tokens:
        return {e: 0.0 for e in EMOTION_LABELS}

    counts = {e: 0 for e in EMOTION_LABELS}
    for w in tokens:
        for e in EMOTION_LABELS:
            if w in FALLBACK_EMO_LEX[e]:
                counts[e] += 1
    denom = max(1, len(tokens))
    return {e: counts[e] / denom for e in EMOTION_LABELS}

def dominant_emotion(emo: Dict[str, float]) -> str:
    if not emo:
        return "none"
    mx = max(emo.values())
    if mx == 0:
        return "none"
    # stable tie-break: alphabetical
    best = sorted([k for k, v in emo.items() if v == mx])[0]
    return best

# ============================================================
# Corpus processing
# ============================================================
@dataclass
class CorpusConfig:
    source_name: str
    text_col: str
    carry_cols: List[str]

def extract_corpus_windows(df: pd.DataFrame, cfg: CorpusConfig, max_rows: Optional[int] = None) -> pd.DataFrame:
    records = []
    df = df.copy()

    if cfg.text_col not in df.columns:
        raise ValueError(f"[{cfg.source_name}] Missing text column: {cfg.text_col}")

    if max_rows is not None:
        df = df.head(max_rows)

    df[cfg.text_col] = df[cfg.text_col].astype(str).map(normalize_text)

    total = len(df)
    for i, row in enumerate(df.itertuples(index=True), start=1):
        idx = row.Index
        text = getattr(row, cfg.text_col)

        if not isinstance(text, str) or len(text) < MIN_TEXT_LEN:
            continue

        raw_windows = extract_windows_for_text(text, WINDOW_CHARS)
        merged_windows = merge_overlapping_windows(raw_windows, gap=0)

        for w_i, (s, e, terms) in enumerate(merged_windows):
            window_text = text[s:e]
            themes = terms_to_categories(terms)
            sent_score, sent_label = vader_sentiment(window_text)
            stance = classify_stance(window_text)

            emo = emotion_scores(window_text)
            dom_emo = dominant_emotion(emo)

            rec = {
                "source": cfg.source_name,
                "record_id": idx,
                "window_id": f"{cfg.source_name}_{idx}_{w_i}",
                "char_start": s,
                "char_end": e,
                "window_text": window_text,
                "keywords_hit": "; ".join(sorted(set(terms))),
                "themes_hit": "; ".join(themes),
                "sentiment_compound": sent_score,
                "sentiment_label": sent_label,
                "stance": stance,
                "dominant_emotion": dom_emo,
                "text_len": len(text),
                "window_len": len(window_text),
            }

            # add emotion columns
            for e_name in EMOTION_LABELS:
                rec[f"emo_{e_name}"] = emo.get(e_name, 0.0)

            # carry columns if exist
            for col in cfg.carry_cols:
                if col in df.columns:
                    rec[col] = df.at[idx, col]

            records.append(rec)

        if (i % 5000) == 0:
            print(f"[{cfg.source_name}] processed {i:,}/{total:,} rows...")

    return pd.DataFrame(records)

# ============================================================
# Plot helpers
# ============================================================
def save_bar_from_pivot(pivot_df: pd.DataFrame, title: str, ylabel: str, outpath: Path, figsize=(7,4)):
    ax = pivot_df.plot(kind="bar", figsize=figsize)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

def save_hbar(df: pd.DataFrame, x: str, y: str, title: str, outpath: Path, figsize=(6,5)):
    plt.figure(figsize=figsize)
    plt.barh(df[y], df[x])
    plt.title(title)
    plt.xlabel(x)
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

def save_heatmap(values: np.ndarray, ylabels: List[str], xlabels: List[str], title: str, outpath: Path, figsize=(7,6)):
    plt.figure(figsize=figsize)
    plt.imshow(values, aspect="auto")
    plt.yticks(range(len(ylabels)), ylabels)
    plt.xticks(range(len(xlabels)), xlabels)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

# ============================================================
# Main pipeline
# ============================================================
def run_pipeline():
    # -------------------------
    # Load
    # -------------------------
    print("Loading Excel files...")
    amazon_df = load_excel(AMAZON_PATH)
    city_df   = load_excel(CITY_PATH)

    amazon_df = canonicalize_columns(amazon_df)
    city_df   = canonicalize_columns(city_df)

    # Detect text cols
    amazon_text_col = find_column(amazon_df, [c.lower() for c in AMAZON_TEXT_COL_CANDIDATES]) or find_column(amazon_df, AMAZON_TEXT_COL_CANDIDATES)
    city_text_col   = find_column(city_df, [c.lower() for c in CITY_TEXT_COL_CANDIDATES]) or find_column(city_df, CITY_TEXT_COL_CANDIDATES)

    if amazon_text_col is None:
        raise ValueError(f"Amazon text column not found. Available: {list(amazon_df.columns)}")
    if city_text_col is None:
        raise ValueError(f"City text column not found. Available: {list(city_df.columns)}")

    # Add city state + year (so state is guaranteed available)
    city_df = add_city_state_columns(city_df)
    city_df = add_city_year_column(city_df)

    # Amazon timestamp->datetime (if possible)
    if "timestamp" in amazon_df.columns:
        ts = pd.to_numeric(amazon_df["timestamp"], errors="coerce")
        if ts.dropna().size > 0:
            median = ts.dropna().median()
            if median > 10**11:
                amazon_df["review_datetime"] = pd.to_datetime(ts, unit="ms", errors="coerce")
            else:
                amazon_df["review_datetime"] = pd.to_datetime(ts, unit="s", errors="coerce")

    # Ensure final outputs always have state fields (amazon will be NaN)
    for col in ["state_raw", "state_abbrev", "state_name", "year"]:
        if col not in amazon_df.columns:
            amazon_df[col] = np.nan

    print(f"Amazon text column: {amazon_text_col}")
    print(f"City text column:   {city_text_col}")

    # -------------------------
    # Extract windows
    # -------------------------
    amazon_cfg = CorpusConfig(
        source_name="amazon",
        text_col=amazon_text_col,
        carry_cols=[
            "rating","asin","parent_asin","user_id","verified_purchase","helpful_vote",
            "timestamp","review_datetime",
            # include (empty) state columns for consistent output
            "state_raw","state_abbrev","state_name"
        ]
    )

    city_cfg = CorpusConfig(
        source_name="city",
        text_col=city_text_col,
        carry_cols=[
            "state_raw","state_abbrev","state_name","year",
            # include these if present in your file
            "city","place_govt","meeting_type","date"
        ]
    )

    print("\nExtracting windows (Amazon)...")
    amazon_windows = extract_corpus_windows(amazon_df, amazon_cfg, max_rows=MAX_ROWS_AMAZON)
    print(f"Amazon windows extracted: {len(amazon_windows):,}")

    print("\nExtracting windows (City)...")
    city_windows = extract_corpus_windows(city_df, city_cfg, max_rows=MAX_ROWS_CITY)
    print(f"City windows extracted: {len(city_windows):,}")

    all_windows = pd.concat([amazon_windows, city_windows], ignore_index=True)

    # -------------------------
    # Summaries
    # -------------------------
    print("\nBuilding summaries...")

    stance_summary = (
        all_windows.groupby(["source","stance"]).size()
        .reset_index(name="n")
        .sort_values(["source","n"], ascending=[True, False])
    )
    stance_summary["pct"] = stance_summary.groupby("source")["n"].transform(lambda x: x / x.sum())

    # Theme explode
    theme_rows = all_windows.copy()
    theme_rows["theme"] = theme_rows["themes_hit"].fillna("").map(lambda s: [t.strip() for t in s.split(";") if t.strip()])
    theme_rows = theme_rows.explode("theme")
    theme_rows = theme_rows[theme_rows["theme"].notna() & (theme_rows["theme"] != "")]

    theme_summary = (
        theme_rows.groupby(["source","theme"]).size()
        .reset_index(name="n")
        .sort_values(["source","n"], ascending=[True, False])
    )
    theme_summary["pct"] = theme_summary.groupby("source")["n"].transform(lambda x: x / x.sum())

    sentiment_by_theme = (
        theme_rows.groupby(["source","theme"])["sentiment_compound"]
        .agg(mean="mean", median="median", n="count")
        .reset_index()
        .sort_values(["source","n"], ascending=[True, False])
    )

    stance_by_theme = (
        theme_rows.groupby(["source","theme","stance"]).size()
        .reset_index(name="n")
        .sort_values(["source","theme","n"], ascending=[True, True, False])
    )

    # Emotion summaries (mean score per corpus)
    emo_cols = [f"emo_{e}" for e in EMOTION_LABELS]
    emotion_summary = (
        all_windows.groupby("source")[emo_cols].mean(numeric_only=True)
        .reset_index()
    )

    # Dominant emotion distribution
    dom_emo_summary = (
        all_windows.groupby(["source","dominant_emotion"]).size()
        .reset_index(name="n")
        .sort_values(["source","n"], ascending=[True, False])
    )
    dom_emo_summary["pct"] = dom_emo_summary.groupby("source")["n"].transform(lambda x: x / x.sum())

    # City stance by state (top states by volume)
    city_only = all_windows[all_windows["source"] == "city"].copy()
    city_state_stance = None
    if len(city_only) > 0 and "state_name" in city_only.columns:
        top_states = (
            city_only["state_name"].fillna("Unknown").value_counts().head(15).index.tolist()
        )
        city_state_stance = (
            city_only[city_only["state_name"].fillna("Unknown").isin(top_states)]
            .groupby(["state_name","stance"]).size()
            .reset_index(name="n")
        )

    # City stance by year
    city_year_stance = None
    if len(city_only) > 0 and "year" in city_only.columns:
        city_only["year"] = pd.to_numeric(city_only["year"], errors="coerce")
        city_year_stance = (
            city_only.dropna(subset=["year"])
            .groupby(["year","stance"]).size()
            .reset_index(name="n")
            .sort_values(["year","n"], ascending=[True, False])
        )

    # Amazon sentiment vs rating (if rating exists)
    amazon_only = all_windows[all_windows["source"] == "amazon"].copy()
    if "rating" in amazon_only.columns:
        amazon_only["rating"] = pd.to_numeric(amazon_only["rating"], errors="coerce")

    # -------------------------
    # Save outputs
    # -------------------------
    extracted_path = OUTPUT_DIR / "extracted_windows.xlsx"
    summary_path   = OUTPUT_DIR / "summary_tables.xlsx"

    print("\nSaving Excel outputs...")
    with pd.ExcelWriter(extracted_path, engine="openpyxl") as writer:
        all_windows.to_excel(writer, index=False, sheet_name="windows")

    with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
        stance_summary.to_excel(writer, index=False, sheet_name="stance_summary")
        theme_summary.to_excel(writer, index=False, sheet_name="theme_summary")
        sentiment_by_theme.to_excel(writer, index=False, sheet_name="sentiment_by_theme")
        stance_by_theme.to_excel(writer, index=False, sheet_name="stance_by_theme")
        emotion_summary.to_excel(writer, index=False, sheet_name="emotion_summary")
        dom_emo_summary.to_excel(writer, index=False, sheet_name="dominant_emotion_summary")
        if city_state_stance is not None:
            city_state_stance.to_excel(writer, index=False, sheet_name="city_state_stance")
        if city_year_stance is not None:
            city_year_stance.to_excel(writer, index=False, sheet_name="city_year_stance")

    print(f"Saved: {extracted_path.resolve()}")
    print(f"Saved: {summary_path.resolve()}")

    # -------------------------
    # Figures
    # -------------------------
    print("\nSaving figures...")

    # 1) Stance distribution by corpus
    fig1 = FIG_DIR / "fig1_stance_distribution_by_corpus.png"
    pivot_stance = stance_summary.pivot(index="stance", columns="source", values="pct").fillna(0.0)
    order = ["support","conditional","oppose","neutral"]
    pivot_stance = pivot_stance.reindex([o for o in order if o in pivot_stance.index])
    save_bar_from_pivot(pivot_stance, "Stance distribution by corpus", "Proportion of windows", fig1)

    # 2) Theme frequency (top 12 each) - horizontal bars
    topN = 12
    for src in ["city","amazon"]:
        sub = theme_summary[theme_summary["source"] == src].head(topN).sort_values("n")
        fig = FIG_DIR / f"fig2_top_themes_{src}.png"
        save_hbar(sub, x="n", y="theme", title=f"Top {topN} themes in {src}", outpath=fig, figsize=(7,5))

    # 3) Sentiment heatmap by theme (mean sentiment) for union of top themes
    fig3 = FIG_DIR / "fig3_mean_sentiment_by_theme_heatmap.png"
    top_city = theme_summary[theme_summary["source"] == "city"].head(topN)["theme"].tolist()
    top_amz  = theme_summary[theme_summary["source"] == "amazon"].head(topN)["theme"].tolist()
    top_union = sorted(set(top_city + top_amz))

    heat = sentiment_by_theme[sentiment_by_theme["theme"].isin(top_union)].copy()
    heat_pivot = heat.pivot(index="theme", columns="source", values="mean").reindex(top_union)
    save_heatmap(
        heat_pivot.fillna(0.0).values,
        ylabels=list(heat_pivot.index),
        xlabels=list(heat_pivot.columns),
        title="Mean sentiment (compound) by theme and corpus",
        outpath=fig3,
        figsize=(7, max(5, len(top_union)*0.35))
    )

    # 4) Emotion profile by corpus (mean emotion scores)
    fig4 = FIG_DIR / "fig4_emotion_profile_by_corpus.png"
    emo_plot = emotion_summary.set_index("source")[emo_cols]
    ax = emo_plot.T.plot(kind="bar", figsize=(9,5))
    ax.set_title("Emotion profile by corpus (mean normalized scores)")
    ax.set_ylabel("Mean emotion score")
    plt.tight_layout()
    plt.savefig(fig4, dpi=200)
    plt.close()

    # 5) Dominant emotion distribution
    fig5 = FIG_DIR / "fig5_dominant_emotion_distribution.png"
    dom_pivot = dom_emo_summary.pivot(index="dominant_emotion", columns="source", values="pct").fillna(0.0)
    ax = dom_pivot.plot(kind="bar", figsize=(9,5))
    ax.set_title("Dominant emotion distribution by corpus")
    ax.set_ylabel("Proportion of windows")
    plt.tight_layout()
    plt.savefig(fig5, dpi=200)
    plt.close()

    # 6) City stance by state (top 15 states) - stacked bars
    if city_state_stance is not None:
        fig6 = FIG_DIR / "fig6_city_stance_by_state_top15.png"
        pivot = city_state_stance.pivot(index="state_name", columns="stance", values="n").fillna(0)
        # normalize to pct within state
        pivot_pct = pivot.div(pivot.sum(axis=1), axis=0)
        pivot_pct = pivot_pct[[c for c in ["support","conditional","oppose","neutral"] if c in pivot_pct.columns]]
        ax = pivot_pct.plot(kind="bar", stacked=True, figsize=(12,6))
        ax.set_title("City corpus: stance distribution by state (top 15 states by volume)")
        ax.set_ylabel("Proportion of windows")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.savefig(fig6, dpi=200)
        plt.close()

    # 7) City stance trend by year (stacked area-like via lines)
    if city_year_stance is not None and len(city_year_stance) > 0:
        fig7 = FIG_DIR / "fig7_city_stance_trend_by_year.png"
        pivot = city_year_stance.pivot(index="year", columns="stance", values="n").fillna(0)
        # normalize to pct by year
        pivot_pct = pivot.div(pivot.sum(axis=1), axis=0)
        pivot_pct = pivot_pct[[c for c in ["support","conditional","oppose","neutral"] if c in pivot_pct.columns]]
        ax = pivot_pct.plot(figsize=(10,5))
        ax.set_title("City corpus: stance trend over time (proportion by year)")
        ax.set_ylabel("Proportion of windows")
        ax.set_xlabel("Year")
        plt.tight_layout()
        plt.savefig(fig7, dpi=200)
        plt.close()

    # 8) Amazon: sentiment vs rating (scatter)
    if "rating" in amazon_only.columns and amazon_only["rating"].notna().any():
        fig8 = FIG_DIR / "fig8_amazon_sentiment_vs_rating.png"
        sub = amazon_only.dropna(subset=["rating"]).copy()
        # downsample for plotting if huge
        if len(sub) > 20000:
            sub = sub.sample(20000, random_state=RANDOM_SEED)
        plt.figure(figsize=(7,5))
        plt.scatter(sub["rating"], sub["sentiment_compound"], alpha=0.3)
        plt.title("Amazon windows: sentiment (compound) vs rating")
        plt.xlabel("Rating")
        plt.ylabel("Sentiment (compound)")
        plt.tight_layout()
        plt.savefig(fig8, dpi=200)
        plt.close()

    print(f"\nFigures saved in: {FIG_DIR.resolve()}")
    print("Pipeline complete.")

    return all_windows


# ============================================================
# RUN
# ============================================================
all_windows = run_pipeline()


Loading Excel files...
Amazon text column: text
City text column:   caption_text_clean

Extracting windows (Amazon)...
[amazon] processed 5,000/138,488 rows...
[amazon] processed 10,000/138,488 rows...
[amazon] processed 15,000/138,488 rows...
[amazon] processed 20,000/138,488 rows...
[amazon] processed 25,000/138,488 rows...
[amazon] processed 30,000/138,488 rows...
[amazon] processed 35,000/138,488 rows...
[amazon] processed 40,000/138,488 rows...
[amazon] processed 45,000/138,488 rows...
[amazon] processed 50,000/138,488 rows...
[amazon] processed 55,000/138,488 rows...
[amazon] processed 60,000/138,488 rows...
[amazon] processed 65,000/138,488 rows...
[amazon] processed 70,000/138,488 rows...
[amazon] processed 75,000/138,488 rows...
[amazon] processed 80,000/138,488 rows...
[amazon] processed 90,000/138,488 rows...
[amazon] processed 105,000/138,488 rows...
[amazon] processed 110,000/138,488 rows...
[amazon] processed 115,000/138,488 rows...
[amazon] processed 120,000/138,488 rows

In [2]:
# ============================================================
# Coherence score testing on ALL extracted windows (Amazon ONLY)
# Uses outputs/extracted_windows-amazon_all_docs.xlsx (sheet: windows)
# ============================================================

from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt

from gensim.utils import simple_preprocess
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models.coherencemodel import CoherenceModel
from gensim.parsing.preprocessing import STOPWORDS as GENSIM_STOPWORDS

# -----------------------------
# Paths
# -----------------------------
WINDOWS_XLSX = Path("outputs") / "extracted_windows.xlsx"
SHEET = "windows"
OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Settings
# -----------------------------
START_TOPICS = 4
END_TOPICS   = 10   # this will evaluate K=4..9 (because range end is exclusive)
STEP         = 1

PASSES = 6
ITERATIONS = 120
CHUNKSIZE = 3000
RANDOM_STATE = 42
WORKERS = max(1, os.cpu_count() - 1)

NO_BELOW = 2
NO_ABOVE = 0.8
KEEP_N   = 100000

EXTRA_STOPWORDS = set([
    "would","could","also","one","two",
    "meeting","council","city","agenda","item",
    "amazon","review","product"
])
STOPWORDS = set(GENSIM_STOPWORDS).union(EXTRA_STOPWORDS)

# -----------------------------
# Preprocess (keeps ALL documents)
# -----------------------------
def preprocess_keep_all(series: pd.Series):
    """
    Returns list-of-token-lists with the SAME number of documents as input.
    If a doc becomes empty, it becomes ["emptydoc"] so it is still included.
    """
    texts = []
    for s in series.fillna("").astype(str).tolist():
        toks = simple_preprocess(s, deacc=True, min_len=2)
        toks = [t for t in toks if t not in STOPWORDS]
        if len(toks) == 0:
            toks = ["emptydoc"]
        texts.append(toks)
    return texts

def build_dictionary_and_corpus(texts):
    dictionary = Dictionary(texts)
    dictionary.filter_extremes(no_below=NO_BELOW, no_above=NO_ABOVE, keep_n=KEEP_N)

    corpus = []
    texts_fixed = []
    for t in texts:
        bow = dictionary.doc2bow(t)
        if len(bow) == 0:
            bow = dictionary.doc2bow(["emptydoc"])
            t = ["emptydoc"]
        corpus.append(bow)
        texts_fixed.append(t)
    return dictionary, corpus, texts_fixed

def coherence_curve(texts, label):
    dictionary, corpus, texts_fixed = build_dictionary_and_corpus(texts)

    print(f"[{label}] Total docs used: {len(texts_fixed):,}")
    print(f"[{label}] Vocab size: {len(dictionary):,}")

    rows = []
    for k in range(START_TOPICS, END_TOPICS, STEP):
        print(f"[{label}] Training LDA K={k} ...")
        model = LdaMulticore(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=RANDOM_STATE,
            chunksize=CHUNKSIZE,
            passes=PASSES,
            iterations=ITERATIONS,
            workers=WORKERS,
            eval_every=None
        )
        cm = CoherenceModel(model=model, texts=texts_fixed, dictionary=dictionary, coherence="c_v")
        coh = cm.get_coherence()
        print(f"[{label}] coherence(c_v)={coh:.4f}")
        rows.append({"source": label, "num_topics": k, "coherence_cv": coh})

    return pd.DataFrame(rows)

# -----------------------------
# Load extracted windows
# -----------------------------
if not WINDOWS_XLSX.exists():
    raise FileNotFoundError(f"Missing file: {WINDOWS_XLSX.resolve()}")

windows = pd.read_excel(WINDOWS_XLSX, sheet_name=SHEET)

if "source" not in windows.columns or "window_text" not in windows.columns:
    raise ValueError("Expected columns: 'source' and 'window_text' in extracted_windows.xlsx")

amazon_text = windows.loc[windows["source"].astype(str).str.lower() == "amazon", "window_text"]

print("Windows total:", len(windows))
print("Amazon windows:", len(amazon_text))

# Preprocess ALL Amazon documents (no dropping)
amazon_tokens = preprocess_keep_all(amazon_text)

# Coherence sweep (Amazon only)
amazon_curve = coherence_curve(amazon_tokens, "amazon")

# Save table
out_xlsx = OUT_DIR / "coherence_results_amazon_all_docs.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    amazon_curve.to_excel(writer, index=False, sheet_name="amazon_curve")

print("Saved:", out_xlsx.resolve())

# Plot (Amazon only)
fig_path = FIG_DIR / "coherence_amazon_all_docs.png"
plt.figure(figsize=(9, 5))
sub = amazon_curve.sort_values("num_topics")
plt.plot(sub["num_topics"], sub["coherence_cv"], marker="o", label="amazon")

plt.title("Coherence (c_v) vs Number of Topics (LDA) — Amazon (ALL docs)")
plt.xlabel("Number of topics (K)")
plt.ylabel("Coherence (c_v)")
plt.xticks(list(range(START_TOPICS, END_TOPICS, STEP)))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(fig_path, dpi=220)
plt.close()

print("Saved plot:", fig_path.resolve())

# Best K
best = amazon_curve.sort_values("coherence_cv", ascending=False).head(1)
print("\nBest K for Amazon (highest c_v):")
print(best.to_string(index=False))


Windows total: 992743
Amazon windows: 48271
[amazon] Total docs used: 48,271
[amazon] Vocab size: 10,557
[amazon] Training LDA K=4 ...
[amazon] coherence(c_v)=0.4700
[amazon] Training LDA K=5 ...
[amazon] coherence(c_v)=0.4757
[amazon] Training LDA K=6 ...
[amazon] coherence(c_v)=0.4833
[amazon] Training LDA K=7 ...
[amazon] coherence(c_v)=0.4793
[amazon] Training LDA K=8 ...
[amazon] coherence(c_v)=0.4791
[amazon] Training LDA K=9 ...
[amazon] coherence(c_v)=0.4764
Saved: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\coherence_results_amazon_all_docs.xlsx
Saved plot: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\figures\coherence_amazon_all_docs.png

Best K for Amazon (highest c_v):
source  num_topics  coherence_cv
amazon           6      0.483276


In [3]:
# ============================================================
# Coherence score testing on ALL extracted windows (CITY ONLY)
# Uses outputs/extracted_windows.xlsx (sheet: windows)
# ============================================================

from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt

from gensim.utils import simple_preprocess
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models.coherencemodel import CoherenceModel
from gensim.parsing.preprocessing import STOPWORDS as GENSIM_STOPWORDS

# -----------------------------
# Paths
# -----------------------------
WINDOWS_XLSX = Path("outputs") / "extracted_windows.xlsx"
SHEET = "windows"
OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Settings
# -----------------------------
START_TOPICS = 4
END_TOPICS   = 10   # evaluates K=4..9 (range end is exclusive)
STEP         = 1

PASSES = 6
ITERATIONS = 120
CHUNKSIZE = 3000
RANDOM_STATE = 42
WORKERS = max(1, os.cpu_count() - 1)

NO_BELOW = 2
NO_ABOVE = 0.8
KEEP_N   = 100000

EXTRA_STOPWORDS = set([
    "would","could","also","one","two",
    "meeting","council","city","agenda","item",
    "amazon","review","product"
])
STOPWORDS = set(GENSIM_STOPWORDS).union(EXTRA_STOPWORDS)

# -----------------------------
# Preprocess (keeps ALL documents)
# -----------------------------
def preprocess_keep_all(series: pd.Series):
    """
    Returns list-of-token-lists with the SAME number of documents as input.
    If a doc becomes empty, it becomes ["emptydoc"] so it is still included.
    """
    texts = []
    for s in series.fillna("").astype(str).tolist():
        toks = simple_preprocess(s, deacc=True, min_len=2)
        toks = [t for t in toks if t not in STOPWORDS]
        if len(toks) == 0:
            toks = ["emptydoc"]
        texts.append(toks)
    return texts

def build_dictionary_and_corpus(texts):
    dictionary = Dictionary(texts)
    dictionary.filter_extremes(no_below=NO_BELOW, no_above=NO_ABOVE, keep_n=KEEP_N)

    corpus = []
    texts_fixed = []
    for t in texts:
        bow = dictionary.doc2bow(t)
        if len(bow) == 0:
            bow = dictionary.doc2bow(["emptydoc"])
            t = ["emptydoc"]
        corpus.append(bow)
        texts_fixed.append(t)
    return dictionary, corpus, texts_fixed

def coherence_curve(texts, label):
    dictionary, corpus, texts_fixed = build_dictionary_and_corpus(texts)

    print(f"[{label}] Total docs used: {len(texts_fixed):,}")
    print(f"[{label}] Vocab size: {len(dictionary):,}")

    rows = []
    for k in range(START_TOPICS, END_TOPICS, STEP):
        print(f"[{label}] Training LDA K={k} ...")
        model = LdaMulticore(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=RANDOM_STATE,
            chunksize=CHUNKSIZE,
            passes=PASSES,
            iterations=ITERATIONS,
            workers=WORKERS,
            eval_every=None
        )
        cm = CoherenceModel(model=model, texts=texts_fixed, dictionary=dictionary, coherence="c_v")
        coh = cm.get_coherence()
        print(f"[{label}] coherence(c_v)={coh:.4f}")
        rows.append({"source": label, "num_topics": k, "coherence_cv": coh})

    return pd.DataFrame(rows)

# -----------------------------
# Load extracted windows
# -----------------------------
if not WINDOWS_XLSX.exists():
    raise FileNotFoundError(f"Missing file: {WINDOWS_XLSX.resolve()}")

windows = pd.read_excel(WINDOWS_XLSX, sheet_name=SHEET)

if "source" not in windows.columns or "window_text" not in windows.columns:
    raise ValueError("Expected columns: 'source' and 'window_text' in extracted_windows.xlsx")

city_text = windows.loc[windows["source"].astype(str).str.lower() == "city", "window_text"]

print("Windows total:", len(windows))
print("City windows:", len(city_text))

# Preprocess ALL City documents (no dropping)
city_tokens = preprocess_keep_all(city_text)

# Coherence sweep (City only)
city_curve = coherence_curve(city_tokens, "city")

# Save table
out_xlsx = OUT_DIR / "coherence_results_city_all_docs.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    city_curve.to_excel(writer, index=False, sheet_name="city_curve")
print("Saved:", out_xlsx.resolve())

# Plot (City only)
fig_path = FIG_DIR / "coherence_city_all_docs.png"
plt.figure(figsize=(9, 5))
sub = city_curve.sort_values("num_topics")
plt.plot(sub["num_topics"], sub["coherence_cv"], marker="o", label="city")

plt.title("Coherence (c_v) vs Number of Topics (LDA) — City (ALL docs)")
plt.xlabel("Number of topics (K)")
plt.ylabel("Coherence (c_v)")
plt.xticks(list(range(START_TOPICS, END_TOPICS, STEP)))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(fig_path, dpi=220)
plt.close()
print("Saved plot:", fig_path.resolve())

# Best K
best = city_curve.sort_values("coherence_cv", ascending=False).head(1)
print("\nBest K for City (highest c_v):")
print(best.to_string(index=False))


Windows total: 992743
City windows: 944472
[city] Total docs used: 944,472
[city] Vocab size: 100,000
[city] Training LDA K=4 ...
[city] coherence(c_v)=0.4617
[city] Training LDA K=5 ...
[city] coherence(c_v)=0.4577
[city] Training LDA K=6 ...
[city] coherence(c_v)=0.4632
[city] Training LDA K=7 ...
[city] coherence(c_v)=0.4690
[city] Training LDA K=8 ...
[city] coherence(c_v)=0.4774
[city] Training LDA K=9 ...
[city] coherence(c_v)=0.4790
Saved: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\coherence_results_city_all_docs.xlsx
Saved plot: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\figures\coherence_city_all_docs.png

Best K for City (highest c_v):
source  num_topics  coherence_cv
  city           9      0.478987


In [ ]:
# ============================================================
# FULL CODE (FIXED): Within-theme subtopic modeling + Keyword co-occurrence clustering
# ✅ Produces BOTH City and Amazon subtheme (within-theme) outputs + plots
# ✅ Fixes the issue where Amazon had only 1 cluster by using looser thresholds for Amazon
#
# INPUT:
#   outputs/extracted_windows.xlsx   (sheet: "windows")
#   Required columns: source, window_text, keywords_hit
#
# OUTPUT:
#   outputs/sustainability_public_opinion_within_theme_and_cooc.xlsx
#     - windows_enriched
#     - theme_summary
#     - within_theme_subtopic_summary
#     - keyword_cluster_summary_city
#     - keyword_cluster_summary_amazon
#     - kw_cluster_prevalence_city
#     - kw_cluster_prevalence_amazon
#
# PLOTS (saved to outputs/figures_discovery/):
#   - {source}_dominant_theme_prevalence.png
#   - within_theme_{source}_{theme}_prevalence.png   (for top themes per source)
#   - {source}_keyword_cluster_sizes.png
#   - {source}_kw_cluster_prevalence.png
#   - plus network stats txt files
#
# REQUIREMENTS:
#   pip install gensim networkx openpyxl
# ============================================================

import os
import re
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Tuple, Set, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- gensim ----------------
from gensim.utils import simple_preprocess
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models.phrases import Phrases, Phraser
from gensim.parsing.preprocessing import STOPWORDS as GENSIM_STOPWORDS

# ---------------- networkx ----------------
import networkx as nx

# ============================================================
# PATHS
# ============================================================
INPUT_XLSX = Path("outputs") / "extracted_windows.xlsx"
INPUT_SHEET = "windows"

OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures_discovery"
OUT_XLSX = OUT_DIR / "sustainability_public_opinion_within_theme_and_cooc.xlsx"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# GLOBAL SETTINGS
# ============================================================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
WORKERS = max(1, os.cpu_count() - 1)

# Within-theme LDA settings (many small models; keep moderate)
LDA_PASSES = 6
LDA_ITERATIONS = 160
LDA_CHUNKSIZE = 1500

# Dictionary filtering (within-theme)
NO_BELOW = 6
NO_ABOVE = 0.55
KEEP_N = 40000

# Bigrams (recommended)
USE_BIGRAMS = True
BIGRAM_MIN_COUNT = 10
BIGRAM_THRESHOLD = 10.0

# Minimum windows per theme to run within-theme topic discovery
MIN_DOCS_PER_THEME_CITY = 600
MIN_DOCS_PER_THEME_AMAZON = 250   # looser for Amazon

# Limit plots per source (fix: separate per source)
MAX_PLOT_THEMES_PER_SOURCE = 8

# Subtopic keyword length
TOP_WORDS_TO_SHOW = 12

# ============================================================
# Co-occurrence thresholds (IMPORTANT: Amazon should be looser)
# ============================================================
COOC_SETTINGS = {
    "city": {
        "min_keyword_freq": 250,
        "min_edge_cooc": 80,
        "max_keywords": 300
    },
    "amazon": {
        "min_keyword_freq": 80,   # loosened to avoid only-one-cluster result
        "min_edge_cooc": 25,      # loosened to retain more edges
        "max_keywords": 300
    }
}

# ============================================================
# KEYWORD DICTIONARY (themes aligned to your paper layout)
# ============================================================
KEYWORDS_BY_CATEGORY: Dict[str, List[str]] = {
    "policy_regulation": [
        "policy", "policies", "regulation", "regulations", "regulatory",
        "law", "laws", "ordinance", "ordinances", "code", "compliance",
        "mandate", "mandated", "requirement", "required", "rule", "rules",
        "standard", "standards", "enforcement", "enforce", "penalty", "penalties",
        "fine", "fines", "citation", "permit", "permitting",
        "initiative", "program", "plan", "strategy", "roadmap",
        "target", "targets", "goal", "goals", "benchmark", "benchmarks",
        "public comment", "public hearing", "stakeholder", "community input"
    ],
    "sustainability_environment": [
        "sustainability", "sustainable",
        "environment", "environmental",
        "climate", "climate action", "climate-friendly",
        "carbon", "carbon footprint", "emissions", "greenhouse gas", "ghg",
        "energy efficient", "energy efficiency",
        "pollution", "conservation", "ecological",
        "eco-friendly", "green product", "green"
    ],
    "circular_design": [
        "circular economy", "circular",
        "lifecycle", "life cycle", "product lifecycle",
        "durability", "durable", "long-lasting", "longevity", "lifespan",
        "repair", "repairable", "repairability", "fixable", "serviceable",
        "right to repair", "repair rights",
        "modular", "modularity",
        "reusable", "reuse", "refill", "refillable",
        "recyclable", "recyclability", "recycle", "recycling",
        "compostable", "compost", "biodegradable",
        "recycled content", "post-consumer recycled", "pcr",
        "eco design", "ecodesign", "design for environment", "dfe",
        "product stewardship", "stewardship",
        "take-back", "takeback", "return-to-manufacturer"
    ],
    "packaging_single_use": [
        "packaging", "packaged", "packaging waste",
        "plastic", "plastics", "single-use", "single use", "disposable",
        "overpackaging", "excess packaging",
        "recyclable packaging", "compostable packaging",
        "paper packaging", "cardboard",
        "bubble wrap", "air pillows", "plastic wrap",
        "bag ban", "plastic bag ban",
        "foam ban", "styrofoam", "polystyrene",
        "container deposit", "bottle bill",
        "refill station", "bulk buying", "bulk refill"
    ],
    "ecommerce_logistics": [
        "e-commerce", "ecommerce",
        "online shopping", "online purchase", "internet retail", "digital marketplace",
        "delivery", "shipping", "shipped",
        "last-mile delivery", "last mile",
        "delivery emissions", "shipping emissions",
        "returns", "return policy", "reverse logistics", "refund", "refunded",
        "warehouse", "fulfillment center",
        "subscription delivery", "recurring delivery"
    ],
    "ewaste_electronics": [
        "electronics", "electronic",
        "e-waste", "ewaste", "electronic waste",
        "device disposal", "dispose of electronics",
        "recycling center", "drop-off", "drop off", "collection event",
        "refurbish", "refurbished", "refurbishment",
        "repair shop", "repair cafe", "repair café",
        "reuse electronics", "device reuse",
        "planned obsolescence", "obsolescence",
        "hazardous waste", "toxic", "heavy metals",
        "lithium battery", "battery recycling", "battery disposal"
    ],
    "labels_trust_transparency": [
        "eco-label", "ecolabel", "eco label",
        "certification", "certified", "third-party certified", "third party certified",
        "epeat", "energy star", "fsc", "fair trade",
        "label claim", "sustainability claim",
        "transparency", "traceability",
        "greenwashing", "misleading", "false claims", "fake", "scam",
        "verified", "verification", "accountability"
    ],
    "consumer_tradeoffs": [
        "affordability", "affordable",
        "expensive", "cost", "costs", "cost increase", "higher price",
        "value", "value for money", "worth it",
        "quality", "low quality", "cheap", "durable quality",
        "convenience", "inconvenient", "hassle", "effort",
        "availability", "hard to find",
        "performance", "functionality", "reliability"
    ],
    "waste_systems": [
        "waste management", "solid waste",
        "landfill", "diversion", "waste diversion",
        "composting program", "organics program",
        "recycling program", "mrf",
        "contamination",
        "collection", "pickup", "curbside",
        "bin", "cart", "trash cart",
        "sanitation department"
    ],
}

TERM_TO_CAT: Dict[str, str] = {}
for cat, terms in KEYWORDS_BY_CATEGORY.items():
    for t in terms:
        TERM_TO_CAT[t.lower().strip()] = cat

# ============================================================
# CLEANING / STOPWORDS (fixes uh, br, mr, okay, thanks...)
# ============================================================
BASE_STOP = set(GENSIM_STOPWORDS)

GLOBAL_NOISE = {
    "br","uh","um","umm","uhh","hmm","mm","mmm","yeah","yep","yes","no","nope","ok","okay","alright","right",
    "mr","mrs","ms","sir","madam","thanks","thank","thankyou","please",
    "applause","laughter","laugh",
    "im","ive","ill","id","weve","theyre","youre","cant","couldnt","dont","doesnt","isnt","wasnt","wont",
    "gonna","wanna","gotta",
    "like","just","really","actually","basically","kind","sort",
    "thing","things","stuff","someone","something",
    "going","go","goes","went",
    "area","areas","place","places",
    "today","tomorrow","yesterday"
}

CITY_PROCEDURE_NOISE = {
    "council","councilmember","councilman","councilwoman","mayor","chair","clerk",
    "meeting","agenda","minutes","item","motion","second","aye","nay","vote","voted",
    "committee","resolution","present","approved","call","order",
    "roll","rollcall","staff","department"
}

AMAZON_REVIEW_NOISE = {
    "amazon","review","reviews","product","products","item","items",
    "buy","bought","purchase","purchased","order","ordered","seller",
    "stars","star","rating"
}

# KEEP packaging-related words (central to your paper)
for w in ["packaging","plastic","shipping","delivery","returns","return","recyclable","recycling"]:
    AMAZON_REVIEW_NOISE.discard(w)
    CITY_PROCEDURE_NOISE.discard(w)

STOP_CITY = BASE_STOP.union(GLOBAL_NOISE).union(CITY_PROCEDURE_NOISE)
STOP_AMAZON = BASE_STOP.union(GLOBAL_NOISE).union(AMAZON_REVIEW_NOISE)

HTML_BR_RE = re.compile(r"<br\s*/?>", re.IGNORECASE)

def clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = HTML_BR_RE.sub(" ", s)
    s = s.replace("&nbsp;", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_docs(texts: List[str], stopwords: Set[str]) -> Tuple[List[List[str]], List[int]]:
    """
    Tokenize ALL docs, return:
      token_lists: tokens for docs kept (>=3 tokens)
      kept_positions: positions in 'texts' that were kept
    """
    token_lists, kept_positions = [], []
    for i, t in enumerate(texts):
        t = clean_text(t).lower()
        toks = simple_preprocess(t, deacc=True, min_len=3)
        toks = [w for w in toks if w not in stopwords and len(w) >= 3]
        if len(toks) >= 3:
            token_lists.append(toks)
            kept_positions.append(i)
    return token_lists, kept_positions

def apply_bigrams(token_lists: List[List[str]]) -> List[List[str]]:
    if not USE_BIGRAMS or len(token_lists) < 200:
        return token_lists
    phrases = Phrases(token_lists, min_count=BIGRAM_MIN_COUNT, threshold=BIGRAM_THRESHOLD)
    phraser = Phraser(phrases)
    return [phraser[toks] for toks in token_lists]

# ============================================================
# THEME ASSIGNMENT
# ============================================================
def parse_keywords_hit(s: str) -> List[str]:
    if not isinstance(s, str):
        return []
    parts = [p.strip().lower() for p in s.split(";")]
    return [p for p in parts if p]

def assign_dominant_theme(keywords: List[str]) -> Tuple[str, float, Dict[str, int]]:
    """
    dominant_theme = category with max keyword hits
    theme_score = max_count / total_hits
    """
    counts: Dict[str, int] = {}
    for kw in keywords:
        cat = TERM_TO_CAT.get(kw)
        if cat:
            counts[cat] = counts.get(cat, 0) + 1
    if not counts:
        return ("none", 0.0, {})
    total = sum(counts.values())
    dom = max(counts.items(), key=lambda x: x[1])[0]
    score = counts[dom] / max(1, total)
    return (dom, score, counts)

# ============================================================
# WITHIN-THEME SUBTOPIC MODELING (LDA per theme)
# ============================================================
def choose_k_for_theme(n_docs: int, max_k: int = 6, min_k: int = 2) -> int:
    # Stable heuristic: grows slowly with theme size, capped
    k = int(np.clip(round(np.sqrt(n_docs / 200)), min_k, max_k))
    return max(min_k, min(max_k, k))

def train_lda(token_lists: List[List[str]], k: int, label: str):
    dictionary = Dictionary(token_lists)
    dictionary.filter_extremes(no_below=NO_BELOW, no_above=NO_ABOVE, keep_n=KEEP_N)
    corpus = [dictionary.doc2bow(toks) for toks in token_lists]

    keep_idx = [i for i, bow in enumerate(corpus) if len(bow) > 0]
    corpus = [corpus[i] for i in keep_idx]
    token_lists = [token_lists[i] for i in keep_idx]

    if len(corpus) == 0 or len(dictionary) == 0:
        return None, None, None, []

    model = LdaMulticore(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=RANDOM_STATE,
        chunksize=LDA_CHUNKSIZE,
        passes=LDA_PASSES,
        iterations=LDA_ITERATIONS,
        workers=WORKERS,
        eval_every=None
    )
    return model, dictionary, corpus, keep_idx

def assign_topics(model, corpus) -> Tuple[np.ndarray, np.ndarray]:
    dom_topic = np.zeros(len(corpus), dtype=int)
    dom_score = np.zeros(len(corpus), dtype=float)
    for i, bow in enumerate(corpus):
        dist = model.get_document_topics(bow, minimum_probability=0.0)
        dist_sorted = sorted(dist, key=lambda x: x[1], reverse=True)
        dom_topic[i] = int(dist_sorted[0][0])
        dom_score[i] = float(dist_sorted[0][1])
    return dom_topic, dom_score

def topic_keywords_str(model, topic_id: int, topn: int = TOP_WORDS_TO_SHOW) -> str:
    return ", ".join([w for w, _ in model.show_topic(topic_id, topn=topn)])

# ============================================================
# KEYWORD CO-OCCURRENCE CLUSTERING (community detection)
# ============================================================
def build_keyword_network(
    keywords_per_window: List[List[str]],
    min_kw_freq: int,
    min_edge_cooc: int,
    max_keywords: int
) -> Tuple[nx.Graph, Dict[str, int]]:
    """
    Graph nodes = keywords (filtered by freq), edges weighted by co-occurrence count.
    """
    freq: Dict[str, int] = {}
    for kws in keywords_per_window:
        uniq = set([k for k in kws if k])
        for k in uniq:
            freq[k] = freq.get(k, 0) + 1

    sorted_k = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    kept = [k for k, c in sorted_k if c >= min_kw_freq][:max_keywords]
    kept_set = set(kept)

    edge_counts: Dict[Tuple[str, str], int] = {}
    for kws in keywords_per_window:
        uniq = sorted(set([k for k in kws if k in kept_set]))
        if len(uniq) < 2:
            continue
        for a, b in combinations(uniq, 2):
            edge_counts[(a, b)] = edge_counts.get((a, b), 0) + 1

    G = nx.Graph()
    for k in kept:
        G.add_node(k, freq=freq[k])

    for (a, b), w in edge_counts.items():
        if w >= min_edge_cooc:
            G.add_edge(a, b, weight=w)

    # remove isolates
    isolates = [n for n in list(G.nodes) if G.degree(n) == 0]
    G.remove_nodes_from(isolates)

    freq2 = {n: int(G.nodes[n].get("freq", 0)) for n in G.nodes}
    return G, freq2

def detect_keyword_communities(G: nx.Graph) -> List[Set[str]]:
    if G.number_of_nodes() == 0:
        return []
    comms = list(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"))
    comms = sorted(comms, key=lambda c: len(c), reverse=True)
    return comms

def label_community(comm: Set[str], freq: Dict[str, int], topn: int = 12) -> str:
    top = sorted(list(comm), key=lambda k: freq.get(k, 0), reverse=True)[:topn]
    return ", ".join(top)

# ============================================================
# PLOTTING HELPERS
# ============================================================
def save_hbar(labels, values, title, xlabel, outpath):
    plt.figure(figsize=(10, max(4, 0.25 * len(labels) + 1)))
    plt.barh(labels, values)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()

def save_bar(x, y, title, xlabel, ylabel, outpath, rotate=False):
    plt.figure(figsize=(10, 4))
    plt.bar(x, y)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if rotate:
        plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()

# ============================================================
# RUN
# ============================================================
if not INPUT_XLSX.exists():
    raise FileNotFoundError(f"Missing input: {INPUT_XLSX.resolve()}")

df = pd.read_excel(INPUT_XLSX, sheet_name=INPUT_SHEET)

required_cols = {"source", "window_text", "keywords_hit"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns {missing} in extracted_windows.xlsx")

df["source"] = df["source"].astype(str).str.lower()
df["window_text"] = df["window_text"].astype(str).map(clean_text)

# Parse keyword hits
df["keywords_list"] = df["keywords_hit"].apply(parse_keywords_hit)

# Dominant theme + score
dom_theme, theme_score = [], []
for kws in df["keywords_list"].tolist():
    d, s, _ = assign_dominant_theme(kws)
    dom_theme.append(d)
    theme_score.append(s)

df["dominant_theme"] = dom_theme
df["theme_score"] = theme_score

# Theme summary
theme_summary = (
    df.groupby(["source", "dominant_theme"]).size()
      .reset_index(name="n")
      .sort_values(["source", "n"], ascending=[True, False])
)
theme_summary["pct"] = theme_summary.groupby("source")["n"].transform(lambda x: x / x.sum())

# Plot: dominant theme prevalence per source
for src in ["amazon", "city"]:
    sub = theme_summary[(theme_summary["source"] == src) & (theme_summary["dominant_theme"] != "none")].head(12)
    save_hbar(
        labels=list(reversed(sub["dominant_theme"].tolist())),
        values=list(reversed(sub["pct"].tolist())),
        title=f"{src}: dominant theme prevalence (top 12)",
        xlabel="Proportion of windows",
        outpath=FIG_DIR / f"{src}_dominant_theme_prevalence.png"
    )

# ============================================================
# (1) WITHIN-THEME SUBTOPIC MODELING
# ============================================================
df["subtopic_id"] = np.nan
df["subtopic_score"] = np.nan
df["subtopic_keywords"] = ""

subtopic_summary_rows = []

# FIX: separate plot budget per source
themes_plotted = {"amazon": 0, "city": 0}

for src in ["amazon", "city"]:
    stopwords = STOP_AMAZON if src == "amazon" else STOP_CITY
    min_docs = MIN_DOCS_PER_THEME_AMAZON if src == "amazon" else MIN_DOCS_PER_THEME_CITY

    src_mask = df["source"] == src
    src_df = df[src_mask].copy()

    # Order themes by prevalence
    theme_order = (
        src_df[src_df["dominant_theme"] != "none"]["dominant_theme"]
        .value_counts()
        .index.tolist()
    )

    for theme in theme_order:
        theme_mask = (df["source"] == src) & (df["dominant_theme"] == theme)
        n_docs = int(theme_mask.sum())
        if n_docs < min_docs:
            continue

        texts = df.loc[theme_mask, "window_text"].tolist()
        token_lists, kept_positions = tokenize_docs(texts, stopwords)
        if USE_BIGRAMS:
            token_lists = apply_bigrams(token_lists)

        if len(token_lists) < min_docs:
            continue

        k = choose_k_for_theme(len(token_lists), max_k=6, min_k=2)
        label = f"{src}::{theme} (K={k})"
        print(f"[Within-theme] {label} | docs={len(token_lists):,}")

        model, dictionary, corpus, keep_idx = train_lda(token_lists, k, label)
        if model is None:
            continue

        dom_t, dom_s = assign_topics(model, corpus)

        # Map back to df indices
        theme_indices = df.loc[theme_mask].index.tolist()

        # kept_positions are positions within theme subset texts that survived tokenization
        # keep_idx are positions within token_lists that survived dictionary filtering
        final_positions = [kept_positions[i] for i in keep_idx]          # positions within theme texts
        final_df_indices = [theme_indices[pos] for pos in final_positions]

        for j, df_i in enumerate(final_df_indices):
            t_id = int(dom_t[j])
            t_sc = float(dom_s[j])
            df.at[df_i, "subtopic_id"] = t_id
            df.at[df_i, "subtopic_score"] = t_sc
            df.at[df_i, "subtopic_keywords"] = topic_keywords_str(model, t_id, TOP_WORDS_TO_SHOW)

        # Summary rows per subtopic
        counts = pd.Series(dom_t).value_counts().reindex(range(k), fill_value=0)
        for t_id in range(k):
            subtopic_summary_rows.append({
                "source": src,
                "dominant_theme": theme,
                "subtopic_id": int(t_id),
                "subtopic_n": int(counts.loc[t_id]),
                "subtopic_pct_within_theme": float(counts.loc[t_id] / max(1, counts.sum())),
                "subtopic_keywords_top12": topic_keywords_str(model, int(t_id), TOP_WORDS_TO_SHOW),
                "k_used": int(k),
                "docs_used": int(len(corpus))
            })

        # Plot a limited number per source (FIXED)
        if themes_plotted[src] < MAX_PLOT_THEMES_PER_SOURCE:
            themes_plotted[src] += 1
            save_bar(
                x=list(range(k)),
                y=counts.values.tolist(),
                title=f"Within-theme subtopics: {src} | {theme} (K={k})",
                xlabel="Subtopic ID",
                ylabel="Dominant-window count",
                outpath=FIG_DIR / f"within_theme_{src}_{theme}_prevalence.png"
            )

within_theme_subtopic_summary = pd.DataFrame(subtopic_summary_rows)

# ============================================================
# (2) KEYWORD CO-OCCURRENCE CLUSTERING
# ============================================================
df["kw_cluster_id"] = np.nan
df["kw_cluster_score"] = np.nan
df["kw_cluster_label"] = ""

cluster_summaries: Dict[str, pd.DataFrame] = {}
cluster_prevalences: Dict[str, pd.DataFrame] = {}

for src in ["amazon", "city"]:
    sub = df[df["source"] == src].copy()
    kw_lists = sub["keywords_list"].tolist()

    settings = COOC_SETTINGS[src]
    G, freq = build_keyword_network(
        kw_lists,
        min_kw_freq=settings["min_keyword_freq"],
        min_edge_cooc=settings["min_edge_cooc"],
        max_keywords=settings["max_keywords"]
    )

    comms = detect_keyword_communities(G)

    # Save stats to txt
    stats_txt = (
        f"{src} keyword network\n"
        f"nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, communities={len(comms)}\n"
        f"min_keyword_freq={settings['min_keyword_freq']}, min_edge_cooc={settings['min_edge_cooc']}, max_keywords={settings['max_keywords']}\n"
    )
    with open(FIG_DIR / f"{src}_keyword_network_stats.txt", "w", encoding="utf-8") as f:
        f.write(stats_txt)

    # Map keyword -> cluster + cluster labels
    kw_to_cluster = {}
    cluster_labels = {}
    cluster_sizes = []
    for cid, comm in enumerate(comms):
        for kw in comm:
            kw_to_cluster[kw] = cid
        cluster_labels[cid] = label_community(comm, freq, topn=12)
        cluster_sizes.append((cid, len(comm)))

    cluster_summary = pd.DataFrame([
        {
            "source": src,
            "kw_cluster_id": int(cid),
            "cluster_size_keywords": int(sz),
            "cluster_label_top_keywords": cluster_labels.get(cid, "")
        }
        for cid, sz in sorted(cluster_sizes, key=lambda x: x[1], reverse=True)
    ])
    cluster_summaries[src] = cluster_summary

    # Assign each window to a dominant keyword-cluster
    for idx, kws in zip(sub.index.tolist(), kw_lists):
        counts = {}
        for kw in set(kws):
            if kw in kw_to_cluster:
                c = kw_to_cluster[kw]
                counts[c] = counts.get(c, 0) + 1
        if not counts:
            continue
        dom_c = max(counts.items(), key=lambda x: x[1])[0]
        score = counts[dom_c] / max(1, sum(counts.values()))

        df.at[idx, "kw_cluster_id"] = int(dom_c)
        df.at[idx, "kw_cluster_score"] = float(score)
        df.at[idx, "kw_cluster_label"] = cluster_labels.get(dom_c, "")

    # Cluster size plot (top 12)
    if len(cluster_summary) > 0:
        top_cs = cluster_summary.head(12)
        save_hbar(
            labels=list(reversed(top_cs["kw_cluster_id"].astype(str).tolist())),
            values=list(reversed(top_cs["cluster_size_keywords"].tolist())),
            title=f"{src}: keyword co-occurrence cluster sizes (top 12)",
            xlabel="Number of keywords in cluster",
            outpath=FIG_DIR / f"{src}_keyword_cluster_sizes.png"
        )

    # Dominant cluster prevalence plot (top 12)
    assigned = df[(df["source"] == src) & (df["kw_cluster_id"].notna())].copy()
    if len(assigned) > 0:
        vc = assigned["kw_cluster_id"].astype(int).value_counts().head(12)
        save_bar(
            x=vc.index.astype(int).tolist(),
            y=vc.values.tolist(),
            title=f"{src}: dominant keyword-cluster prevalence (top 12 clusters)",
            xlabel="kw_cluster_id",
            ylabel="Window count",
            outpath=FIG_DIR / f"{src}_kw_cluster_prevalence.png"
        )

        cluster_prevalences[src] = vc.reset_index().rename(columns={"index": "kw_cluster_id", "kw_cluster_id": "n"}).assign(source=src)

# ============================================================
# SAVE OUTPUT EXCEL
# ============================================================
# Clean up: optional drop internal list column to keep excel clean
df_out = df.drop(columns=["keywords_list"], errors="ignore")

kw_prev_city = cluster_prevalences.get("city", pd.DataFrame(columns=["kw_cluster_id", "n", "source"]))
kw_prev_amz  = cluster_prevalences.get("amazon", pd.DataFrame(columns=["kw_cluster_id", "n", "source"]))

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df_out.to_excel(writer, index=False, sheet_name="windows_enriched")
    theme_summary.to_excel(writer, index=False, sheet_name="theme_summary")
    within_theme_subtopic_summary.to_excel(writer, index=False, sheet_name="within_theme_subtopic_summary")
    if "city" in cluster_summaries:
        cluster_summaries["city"].to_excel(writer, index=False, sheet_name="keyword_cluster_summary_city")
    if "amazon" in cluster_summaries:
        cluster_summaries["amazon"].to_excel(writer, index=False, sheet_name="keyword_cluster_summary_amazon")
    kw_prev_city.to_excel(writer, index=False, sheet_name="kw_cluster_prevalence_city")
    kw_prev_amz.to_excel(writer, index=False, sheet_name="kw_cluster_prevalence_amazon")

print("\nDONE ")
print(f"Saved Excel: {OUT_XLSX.resolve()}")
print(f"Saved figures + stats: {FIG_DIR.resolve()}")

# ============================================================
# QUICK DIAGNOSTIC PRINTS (helps confirm Amazon subthemes exist)
# ============================================================
print("\nWithin-theme subtopic summary counts by source:")
if len(within_theme_subtopic_summary) == 0:
    print("  (No within-theme models ran. Lower MIN_DOCS thresholds.)")
else:
    print(within_theme_subtopic_summary["source"].value_counts().to_string())

print("\nKeyword co-occurrence communities by source:")
for src in ["amazon", "city"]:
    stats_file = FIG_DIR / f"{src}_keyword_network_stats.txt"
    if stats_file.exists():
        print(f"\n--- {src} network stats ---")
        print(stats_file.read_text(encoding="utf-8"))

[Within-theme] amazon::consumer_tradeoffs (K=6) | docs=39,246
[Within-theme] amazon::policy_regulation (K=4) | docs=3,554
[Within-theme] amazon::ecommerce_logistics (K=3) | docs=1,535
[Within-theme] amazon::packaging_single_use (K=3) | docs=1,491
[Within-theme] amazon::circular_design (K=2) | docs=954
[Within-theme] amazon::labels_trust_transparency (K=2) | docs=453
[Within-theme] amazon::ewaste_electronics (K=2) | docs=372
[Within-theme] amazon::sustainability_environment (K=2) | docs=279
[Within-theme] city::policy_regulation (K=6) | docs=625,335


In [1]:
# ============================================================
# FULL CODE (FIXED): Within-theme subtopic modeling + Keyword co-occurrence clustering
# ✅ Produces BOTH City and Amazon subtheme (within-theme) outputs + plots
# ✅ Fixes the issue where Amazon had only 1 cluster by using looser thresholds for Amazon
#
# INPUT:
#   outputs/extracted_windows.xlsx   (sheet: "windows")
#   Required columns: source, window_text, keywords_hit
#
# OUTPUT:
#   outputs/sustainability_public_opinion_within_theme_and_cooc.xlsx
#     - windows_enriched
#     - theme_summary
#     - within_theme_subtopic_summary
#     - keyword_cluster_summary_city
#     - keyword_cluster_summary_amazon
#     - kw_cluster_prevalence_city
#     - kw_cluster_prevalence_amazon
#
# PLOTS (saved to outputs/figures_discovery/):
#   - {source}_dominant_theme_prevalence.png
#   - within_theme_{source}_{theme}_prevalence.png   (for top themes per source)
#   - {source}_keyword_cluster_sizes.png
#   - {source}_kw_cluster_prevalence.png
#   - plus network stats txt files
#
# REQUIREMENTS:
#   pip install gensim networkx openpyxl
# ============================================================

import os
import re
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Tuple, Set, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- gensim ----------------
from gensim.utils import simple_preprocess
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models.phrases import Phrases, Phraser
from gensim.parsing.preprocessing import STOPWORDS as GENSIM_STOPWORDS

# ---------------- networkx ----------------
import networkx as nx

# ============================================================
# PATHS
# ============================================================
INPUT_XLSX = Path("outputs") / "extracted_windows.xlsx"
INPUT_SHEET = "windows"

OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures_discovery"
OUT_XLSX = OUT_DIR / "sustainability_public_opinion_within_theme_and_cooc.xlsx"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# GLOBAL SETTINGS
# ============================================================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
WORKERS = max(1, os.cpu_count() - 1)

# Within-theme LDA settings (many small models; keep moderate)
LDA_PASSES = 6
LDA_ITERATIONS = 160
LDA_CHUNKSIZE = 1500

# Dictionary filtering (within-theme)
NO_BELOW = 6
NO_ABOVE = 0.55
KEEP_N = 40000

# Bigrams (recommended)
USE_BIGRAMS = True
BIGRAM_MIN_COUNT = 10
BIGRAM_THRESHOLD = 10.0

# Minimum windows per theme to run within-theme topic discovery
MIN_DOCS_PER_THEME_CITY = 600
MIN_DOCS_PER_THEME_AMAZON = 250   # looser for Amazon

# Limit plots per source (fix: separate per source)
MAX_PLOT_THEMES_PER_SOURCE = 8

# Subtopic keyword length
TOP_WORDS_TO_SHOW = 12

# ============================================================
# Co-occurrence thresholds (IMPORTANT: Amazon should be looser)
# ============================================================
COOC_SETTINGS = {
    "city": {
        "min_keyword_freq": 250,
        "min_edge_cooc": 80,
        "max_keywords": 300
    },
    "amazon": {
        "min_keyword_freq": 80,   # loosened to avoid only-one-cluster result
        "min_edge_cooc": 25,      # loosened to retain more edges
        "max_keywords": 300
    }
}

# ============================================================
# KEYWORD DICTIONARY (themes aligned to your paper layout)
# ============================================================
KEYWORDS_BY_CATEGORY: Dict[str, List[str]] = {
    "policy_regulation": [
        "policy", "policies", "regulation", "regulations", "regulatory",
        "law", "laws", "ordinance", "ordinances", "code", "compliance",
        "mandate", "mandated", "requirement", "required", "rule", "rules",
        "standard", "standards", "enforcement", "enforce", "penalty", "penalties",
        "fine", "fines", "citation", "permit", "permitting",
        "initiative", "program", "plan", "strategy", "roadmap",
        "target", "targets", "goal", "goals", "benchmark", "benchmarks",
        "public comment", "public hearing", "stakeholder", "community input"
    ],
    "sustainability_environment": [
        "sustainability", "sustainable",
        "environment", "environmental",
        "climate", "climate action", "climate-friendly",
        "carbon", "carbon footprint", "emissions", "greenhouse gas", "ghg",
        "energy efficient", "energy efficiency",
        "pollution", "conservation", "ecological",
        "eco-friendly", "green product", "green"
    ],
    "circular_design": [
        "circular economy", "circular",
        "lifecycle", "life cycle", "product lifecycle",
        "durability", "durable", "long-lasting", "longevity", "lifespan",
        "repair", "repairable", "repairability", "fixable", "serviceable",
        "right to repair", "repair rights",
        "modular", "modularity",
        "reusable", "reuse", "refill", "refillable",
        "recyclable", "recyclability", "recycle", "recycling",
        "compostable", "compost", "biodegradable",
        "recycled content", "post-consumer recycled", "pcr",
        "eco design", "ecodesign", "design for environment", "dfe",
        "product stewardship", "stewardship",
        "take-back", "takeback", "return-to-manufacturer"
    ],
    "packaging_single_use": [
        "packaging", "packaged", "packaging waste",
        "plastic", "plastics", "single-use", "single use", "disposable",
        "overpackaging", "excess packaging",
        "recyclable packaging", "compostable packaging",
        "paper packaging", "cardboard",
        "bubble wrap", "air pillows", "plastic wrap",
        "bag ban", "plastic bag ban",
        "foam ban", "styrofoam", "polystyrene",
        "container deposit", "bottle bill",
        "refill station", "bulk buying", "bulk refill"
    ],
    "ecommerce_logistics": [
        "e-commerce", "ecommerce",
        "online shopping", "online purchase", "internet retail", "digital marketplace",
        "delivery", "shipping", "shipped",
        "last-mile delivery", "last mile",
        "delivery emissions", "shipping emissions",
        "returns", "return policy", "reverse logistics", "refund", "refunded",
        "warehouse", "fulfillment center",
        "subscription delivery", "recurring delivery"
    ],
    "ewaste_electronics": [
        "electronics", "electronic",
        "e-waste", "ewaste", "electronic waste",
        "device disposal", "dispose of electronics",
        "recycling center", "drop-off", "drop off", "collection event",
        "refurbish", "refurbished", "refurbishment",
        "repair shop", "repair cafe", "repair café",
        "reuse electronics", "device reuse",
        "planned obsolescence", "obsolescence",
        "hazardous waste", "toxic", "heavy metals",
        "lithium battery", "battery recycling", "battery disposal"
    ],
    "labels_trust_transparency": [
        "eco-label", "ecolabel", "eco label",
        "certification", "certified", "third-party certified", "third party certified",
        "epeat", "energy star", "fsc", "fair trade",
        "label claim", "sustainability claim",
        "transparency", "traceability",
        "greenwashing", "misleading", "false claims", "fake", "scam",
        "verified", "verification", "accountability"
    ],
    "consumer_tradeoffs": [
        "affordability", "affordable",
        "expensive", "cost", "costs", "cost increase", "higher price",
        "value", "value for money", "worth it",
        "quality", "low quality", "cheap", "durable quality",
        "convenience", "inconvenient", "hassle", "effort",
        "availability", "hard to find",
        "performance", "functionality", "reliability"
    ],
    "waste_systems": [
        "waste management", "solid waste",
        "landfill", "diversion", "waste diversion",
        "composting program", "organics program",
        "recycling program", "mrf",
        "contamination",
        "collection", "pickup", "curbside",
        "bin", "cart", "trash cart",
        "sanitation department"
    ],
}

TERM_TO_CAT: Dict[str, str] = {}
for cat, terms in KEYWORDS_BY_CATEGORY.items():
    for t in terms:
        TERM_TO_CAT[t.lower().strip()] = cat

# ============================================================
# CLEANING / STOPWORDS (fixes uh, br, mr, okay, thanks...)
# ============================================================
BASE_STOP = set(GENSIM_STOPWORDS)

GLOBAL_NOISE = {
    "br","uh","um","umm","uhh","hmm","mm","mmm","yeah","yep","yes","no","nope","ok","okay","alright","right",
    "mr","mrs","ms","sir","madam","thanks","thank","thankyou","please",
    "applause","laughter","laugh",
    "im","ive","ill","id","weve","theyre","youre","cant","couldnt","dont","doesnt","isnt","wasnt","wont",
    "gonna","wanna","gotta",
    "like","just","really","actually","basically","kind","sort",
    "thing","things","stuff","someone","something",
    "going","go","goes","went",
    "area","areas","place","places",
    "today","tomorrow","yesterday"
}

CITY_PROCEDURE_NOISE = {
    "council","councilmember","councilman","councilwoman","mayor","chair","clerk",
    "meeting","agenda","minutes","item","motion","second","aye","nay","vote","voted",
    "committee","resolution","present","approved","call","order",
    "roll","rollcall","staff","department"
}

AMAZON_REVIEW_NOISE = {
    "amazon","review","reviews","product","products","item","items",
    "buy","bought","purchase","purchased","order","ordered","seller",
    "stars","star","rating"
}

# KEEP packaging-related words (central to your paper)
for w in ["packaging","plastic","shipping","delivery","returns","return","recyclable","recycling"]:
    AMAZON_REVIEW_NOISE.discard(w)
    CITY_PROCEDURE_NOISE.discard(w)

STOP_CITY = BASE_STOP.union(GLOBAL_NOISE).union(CITY_PROCEDURE_NOISE)
STOP_AMAZON = BASE_STOP.union(GLOBAL_NOISE).union(AMAZON_REVIEW_NOISE)

HTML_BR_RE = re.compile(r"<br\s*/?>", re.IGNORECASE)

def clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = HTML_BR_RE.sub(" ", s)
    s = s.replace("&nbsp;", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_docs(texts: List[str], stopwords: Set[str]) -> Tuple[List[List[str]], List[int]]:
    """
    Tokenize ALL docs, return:
      token_lists: tokens for docs kept (>=3 tokens)
      kept_positions: positions in 'texts' that were kept
    """
    token_lists, kept_positions = [], []
    for i, t in enumerate(texts):
        t = clean_text(t).lower()
        toks = simple_preprocess(t, deacc=True, min_len=3)
        toks = [w for w in toks if w not in stopwords and len(w) >= 3]
        if len(toks) >= 3:
            token_lists.append(toks)
            kept_positions.append(i)
    return token_lists, kept_positions

def apply_bigrams(token_lists: List[List[str]]) -> List[List[str]]:
    if not USE_BIGRAMS or len(token_lists) < 200:
        return token_lists
    phrases = Phrases(token_lists, min_count=BIGRAM_MIN_COUNT, threshold=BIGRAM_THRESHOLD)
    phraser = Phraser(phrases)
    return [phraser[toks] for toks in token_lists]

# ============================================================
# THEME ASSIGNMENT
# ============================================================
def parse_keywords_hit(s: str) -> List[str]:
    if not isinstance(s, str):
        return []
    parts = [p.strip().lower() for p in s.split(";")]
    return [p for p in parts if p]

def assign_dominant_theme(keywords: List[str]) -> Tuple[str, float, Dict[str, int]]:
    """
    dominant_theme = category with max keyword hits
    theme_score = max_count / total_hits
    """
    counts: Dict[str, int] = {}
    for kw in keywords:
        cat = TERM_TO_CAT.get(kw)
        if cat:
            counts[cat] = counts.get(cat, 0) + 1
    if not counts:
        return ("none", 0.0, {})
    total = sum(counts.values())
    dom = max(counts.items(), key=lambda x: x[1])[0]
    score = counts[dom] / max(1, total)
    return (dom, score, counts)

# ============================================================
# WITHIN-THEME SUBTOPIC MODELING (LDA per theme)
# ============================================================
def choose_k_for_theme(n_docs: int, max_k: int = 6, min_k: int = 2) -> int:
    # Stable heuristic: grows slowly with theme size, capped
    k = int(np.clip(round(np.sqrt(n_docs / 200)), min_k, max_k))
    return max(min_k, min(max_k, k))

def train_lda(token_lists: List[List[str]], k: int, label: str):
    dictionary = Dictionary(token_lists)
    dictionary.filter_extremes(no_below=NO_BELOW, no_above=NO_ABOVE, keep_n=KEEP_N)
    corpus = [dictionary.doc2bow(toks) for toks in token_lists]

    keep_idx = [i for i, bow in enumerate(corpus) if len(bow) > 0]
    corpus = [corpus[i] for i in keep_idx]
    token_lists = [token_lists[i] for i in keep_idx]

    if len(corpus) == 0 or len(dictionary) == 0:
        return None, None, None, []

    model = LdaMulticore(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=RANDOM_STATE,
        chunksize=LDA_CHUNKSIZE,
        passes=LDA_PASSES,
        iterations=LDA_ITERATIONS,
        workers=WORKERS,
        eval_every=None
    )
    return model, dictionary, corpus, keep_idx

def assign_topics(model, corpus) -> Tuple[np.ndarray, np.ndarray]:
    dom_topic = np.zeros(len(corpus), dtype=int)
    dom_score = np.zeros(len(corpus), dtype=float)
    for i, bow in enumerate(corpus):
        dist = model.get_document_topics(bow, minimum_probability=0.0)
        dist_sorted = sorted(dist, key=lambda x: x[1], reverse=True)
        dom_topic[i] = int(dist_sorted[0][0])
        dom_score[i] = float(dist_sorted[0][1])
    return dom_topic, dom_score

def topic_keywords_str(model, topic_id: int, topn: int = TOP_WORDS_TO_SHOW) -> str:
    return ", ".join([w for w, _ in model.show_topic(topic_id, topn=topn)])

# ============================================================
# KEYWORD CO-OCCURRENCE CLUSTERING (community detection)
# ============================================================
def build_keyword_network(
    keywords_per_window: List[List[str]],
    min_kw_freq: int,
    min_edge_cooc: int,
    max_keywords: int
) -> Tuple[nx.Graph, Dict[str, int]]:
    """
    Graph nodes = keywords (filtered by freq), edges weighted by co-occurrence count.
    """
    freq: Dict[str, int] = {}
    for kws in keywords_per_window:
        uniq = set([k for k in kws if k])
        for k in uniq:
            freq[k] = freq.get(k, 0) + 1

    sorted_k = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    kept = [k for k, c in sorted_k if c >= min_kw_freq][:max_keywords]
    kept_set = set(kept)

    edge_counts: Dict[Tuple[str, str], int] = {}
    for kws in keywords_per_window:
        uniq = sorted(set([k for k in kws if k in kept_set]))
        if len(uniq) < 2:
            continue
        for a, b in combinations(uniq, 2):
            edge_counts[(a, b)] = edge_counts.get((a, b), 0) + 1

    G = nx.Graph()
    for k in kept:
        G.add_node(k, freq=freq[k])

    for (a, b), w in edge_counts.items():
        if w >= min_edge_cooc:
            G.add_edge(a, b, weight=w)

    # remove isolates
    isolates = [n for n in list(G.nodes) if G.degree(n) == 0]
    G.remove_nodes_from(isolates)

    freq2 = {n: int(G.nodes[n].get("freq", 0)) for n in G.nodes}
    return G, freq2

def detect_keyword_communities(G: nx.Graph) -> List[Set[str]]:
    if G.number_of_nodes() == 0:
        return []
    comms = list(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"))
    comms = sorted(comms, key=lambda c: len(c), reverse=True)
    return comms

def label_community(comm: Set[str], freq: Dict[str, int], topn: int = 12) -> str:
    top = sorted(list(comm), key=lambda k: freq.get(k, 0), reverse=True)[:topn]
    return ", ".join(top)

# ============================================================
# PLOTTING HELPERS
# ============================================================
def save_hbar(labels, values, title, xlabel, outpath):
    plt.figure(figsize=(10, max(4, 0.25 * len(labels) + 1)))
    plt.barh(labels, values)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()

def save_bar(x, y, title, xlabel, ylabel, outpath, rotate=False):
    plt.figure(figsize=(10, 4))
    plt.bar(x, y)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if rotate:
        plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()

# ============================================================
# RUN
# ============================================================
if not INPUT_XLSX.exists():
    raise FileNotFoundError(f"Missing input: {INPUT_XLSX.resolve()}")

df = pd.read_excel(INPUT_XLSX, sheet_name=INPUT_SHEET)

required_cols = {"source", "window_text", "keywords_hit"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns {missing} in extracted_windows.xlsx")

df["source"] = df["source"].astype(str).str.lower()
df["window_text"] = df["window_text"].astype(str).map(clean_text)

# Parse keyword hits
df["keywords_list"] = df["keywords_hit"].apply(parse_keywords_hit)

# Dominant theme + score
dom_theme, theme_score = [], []
for kws in df["keywords_list"].tolist():
    d, s, _ = assign_dominant_theme(kws)
    dom_theme.append(d)
    theme_score.append(s)

df["dominant_theme"] = dom_theme
df["theme_score"] = theme_score

# Theme summary
theme_summary = (
    df.groupby(["source", "dominant_theme"]).size()
      .reset_index(name="n")
      .sort_values(["source", "n"], ascending=[True, False])
)
theme_summary["pct"] = theme_summary.groupby("source")["n"].transform(lambda x: x / x.sum())

# Plot: dominant theme prevalence per source
for src in ["amazon", "city"]:
    sub = theme_summary[(theme_summary["source"] == src) & (theme_summary["dominant_theme"] != "none")].head(12)
    save_hbar(
        labels=list(reversed(sub["dominant_theme"].tolist())),
        values=list(reversed(sub["pct"].tolist())),
        title=f"{src}: dominant theme prevalence (top 12)",
        xlabel="Proportion of windows",
        outpath=FIG_DIR / f"{src}_dominant_theme_prevalence.png"
    )

# ============================================================
# (1) WITHIN-THEME SUBTOPIC MODELING
# ============================================================
df["subtopic_id"] = np.nan
df["subtopic_score"] = np.nan
df["subtopic_keywords"] = ""

subtopic_summary_rows = []

# FIX: separate plot budget per source
themes_plotted = {"amazon": 0, "city": 0}

for src in ["amazon", "city"]:
    stopwords = STOP_AMAZON if src == "amazon" else STOP_CITY
    min_docs = MIN_DOCS_PER_THEME_AMAZON if src == "amazon" else MIN_DOCS_PER_THEME_CITY

    src_mask = df["source"] == src
    src_df = df[src_mask].copy()

    # Order themes by prevalence
    theme_order = (
        src_df[src_df["dominant_theme"] != "none"]["dominant_theme"]
        .value_counts()
        .index.tolist()
    )

    for theme in theme_order:
        theme_mask = (df["source"] == src) & (df["dominant_theme"] == theme)
        n_docs = int(theme_mask.sum())
        if n_docs < min_docs:
            continue

        texts = df.loc[theme_mask, "window_text"].tolist()
        token_lists, kept_positions = tokenize_docs(texts, stopwords)
        if USE_BIGRAMS:
            token_lists = apply_bigrams(token_lists)

        if len(token_lists) < min_docs:
            continue

        k = choose_k_for_theme(len(token_lists), max_k=6, min_k=2)
        label = f"{src}::{theme} (K={k})"
        print(f"[Within-theme] {label} | docs={len(token_lists):,}")

        model, dictionary, corpus, keep_idx = train_lda(token_lists, k, label)
        if model is None:
            continue

        dom_t, dom_s = assign_topics(model, corpus)

        # Map back to df indices
        theme_indices = df.loc[theme_mask].index.tolist()

        # kept_positions are positions within theme subset texts that survived tokenization
        # keep_idx are positions within token_lists that survived dictionary filtering
        final_positions = [kept_positions[i] for i in keep_idx]          # positions within theme texts
        final_df_indices = [theme_indices[pos] for pos in final_positions]

        for j, df_i in enumerate(final_df_indices):
            t_id = int(dom_t[j])
            t_sc = float(dom_s[j])
            df.at[df_i, "subtopic_id"] = t_id
            df.at[df_i, "subtopic_score"] = t_sc
            df.at[df_i, "subtopic_keywords"] = topic_keywords_str(model, t_id, TOP_WORDS_TO_SHOW)

        # Summary rows per subtopic
        counts = pd.Series(dom_t).value_counts().reindex(range(k), fill_value=0)
        for t_id in range(k):
            subtopic_summary_rows.append({
                "source": src,
                "dominant_theme": theme,
                "subtopic_id": int(t_id),
                "subtopic_n": int(counts.loc[t_id]),
                "subtopic_pct_within_theme": float(counts.loc[t_id] / max(1, counts.sum())),
                "subtopic_keywords_top12": topic_keywords_str(model, int(t_id), TOP_WORDS_TO_SHOW),
                "k_used": int(k),
                "docs_used": int(len(corpus))
            })

        # Plot a limited number per source (FIXED)
        if themes_plotted[src] < MAX_PLOT_THEMES_PER_SOURCE:
            themes_plotted[src] += 1
            save_bar(
                x=list(range(k)),
                y=counts.values.tolist(),
                title=f"Within-theme subtopics: {src} | {theme} (K={k})",
                xlabel="Subtopic ID",
                ylabel="Dominant-window count",
                outpath=FIG_DIR / f"within_theme_{src}_{theme}_prevalence.png"
            )

within_theme_subtopic_summary = pd.DataFrame(subtopic_summary_rows)

# ============================================================
# (2) KEYWORD CO-OCCURRENCE CLUSTERING
# ============================================================
df["kw_cluster_id"] = np.nan
df["kw_cluster_score"] = np.nan
df["kw_cluster_label"] = ""

cluster_summaries: Dict[str, pd.DataFrame] = {}
cluster_prevalences: Dict[str, pd.DataFrame] = {}

for src in ["amazon", "city"]:
    sub = df[df["source"] == src].copy()
    kw_lists = sub["keywords_list"].tolist()

    settings = COOC_SETTINGS[src]
    G, freq = build_keyword_network(
        kw_lists,
        min_kw_freq=settings["min_keyword_freq"],
        min_edge_cooc=settings["min_edge_cooc"],
        max_keywords=settings["max_keywords"]
    )

    comms = detect_keyword_communities(G)

    # Save stats to txt
    stats_txt = (
        f"{src} keyword network\n"
        f"nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, communities={len(comms)}\n"
        f"min_keyword_freq={settings['min_keyword_freq']}, min_edge_cooc={settings['min_edge_cooc']}, max_keywords={settings['max_keywords']}\n"
    )
    with open(FIG_DIR / f"{src}_keyword_network_stats.txt", "w", encoding="utf-8") as f:
        f.write(stats_txt)

    # Map keyword -> cluster + cluster labels
    kw_to_cluster = {}
    cluster_labels = {}
    cluster_sizes = []
    for cid, comm in enumerate(comms):
        for kw in comm:
            kw_to_cluster[kw] = cid
        cluster_labels[cid] = label_community(comm, freq, topn=12)
        cluster_sizes.append((cid, len(comm)))

    cluster_summary = pd.DataFrame([
        {
            "source": src,
            "kw_cluster_id": int(cid),
            "cluster_size_keywords": int(sz),
            "cluster_label_top_keywords": cluster_labels.get(cid, "")
        }
        for cid, sz in sorted(cluster_sizes, key=lambda x: x[1], reverse=True)
    ])
    cluster_summaries[src] = cluster_summary

    # Assign each window to a dominant keyword-cluster
    for idx, kws in zip(sub.index.tolist(), kw_lists):
        counts = {}
        for kw in set(kws):
            if kw in kw_to_cluster:
                c = kw_to_cluster[kw]
                counts[c] = counts.get(c, 0) + 1
        if not counts:
            continue
        dom_c = max(counts.items(), key=lambda x: x[1])[0]
        score = counts[dom_c] / max(1, sum(counts.values()))

        df.at[idx, "kw_cluster_id"] = int(dom_c)
        df.at[idx, "kw_cluster_score"] = float(score)
        df.at[idx, "kw_cluster_label"] = cluster_labels.get(dom_c, "")

    # Cluster size plot (top 12)
    if len(cluster_summary) > 0:
        top_cs = cluster_summary.head(12)
        save_hbar(
            labels=list(reversed(top_cs["kw_cluster_id"].astype(str).tolist())),
            values=list(reversed(top_cs["cluster_size_keywords"].tolist())),
            title=f"{src}: keyword co-occurrence cluster sizes (top 12)",
            xlabel="Number of keywords in cluster",
            outpath=FIG_DIR / f"{src}_keyword_cluster_sizes.png"
        )

    # Dominant cluster prevalence plot (top 12)
    assigned = df[(df["source"] == src) & (df["kw_cluster_id"].notna())].copy()
    if len(assigned) > 0:
        vc = assigned["kw_cluster_id"].astype(int).value_counts().head(12)
        save_bar(
            x=vc.index.astype(int).tolist(),
            y=vc.values.tolist(),
            title=f"{src}: dominant keyword-cluster prevalence (top 12 clusters)",
            xlabel="kw_cluster_id",
            ylabel="Window count",
            outpath=FIG_DIR / f"{src}_kw_cluster_prevalence.png"
        )

        cluster_prevalences[src] = vc.reset_index().rename(columns={"index": "kw_cluster_id", "kw_cluster_id": "n"}).assign(source=src)

# ============================================================
# SAVE OUTPUT EXCEL
# ============================================================
# Clean up: optional drop internal list column to keep excel clean
df_out = df.drop(columns=["keywords_list"], errors="ignore")

kw_prev_city = cluster_prevalences.get("city", pd.DataFrame(columns=["kw_cluster_id", "n", "source"]))
kw_prev_amz  = cluster_prevalences.get("amazon", pd.DataFrame(columns=["kw_cluster_id", "n", "source"]))

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df_out.to_excel(writer, index=False, sheet_name="windows_enriched")
    theme_summary.to_excel(writer, index=False, sheet_name="theme_summary")
    within_theme_subtopic_summary.to_excel(writer, index=False, sheet_name="within_theme_subtopic_summary")
    if "city" in cluster_summaries:
        cluster_summaries["city"].to_excel(writer, index=False, sheet_name="keyword_cluster_summary_city")
    if "amazon" in cluster_summaries:
        cluster_summaries["amazon"].to_excel(writer, index=False, sheet_name="keyword_cluster_summary_amazon")
    kw_prev_city.to_excel(writer, index=False, sheet_name="kw_cluster_prevalence_city")
    kw_prev_amz.to_excel(writer, index=False, sheet_name="kw_cluster_prevalence_amazon")

print("\nDONE ")
print(f"Saved Excel: {OUT_XLSX.resolve()}")
print(f"Saved figures + stats: {FIG_DIR.resolve()}")

# ============================================================
# QUICK DIAGNOSTIC PRINTS (helps confirm Amazon subthemes exist)
# ============================================================
print("\nWithin-theme subtopic summary counts by source:")
if len(within_theme_subtopic_summary) == 0:
    print("  (No within-theme models ran. Lower MIN_DOCS thresholds.)")
else:
    print(within_theme_subtopic_summary["source"].value_counts().to_string())

print("\nKeyword co-occurrence communities by source:")
for src in ["amazon", "city"]:
    stats_file = FIG_DIR / f"{src}_keyword_network_stats.txt"
    if stats_file.exists():
        print(f"\n--- {src} network stats ---")
        print(stats_file.read_text(encoding="utf-8"))

[Within-theme] amazon::consumer_tradeoffs (K=6) | docs=39,246
[Within-theme] amazon::policy_regulation (K=4) | docs=3,554
[Within-theme] amazon::ecommerce_logistics (K=3) | docs=1,535
[Within-theme] amazon::packaging_single_use (K=3) | docs=1,491
[Within-theme] amazon::circular_design (K=2) | docs=954
[Within-theme] amazon::labels_trust_transparency (K=2) | docs=453
[Within-theme] amazon::ewaste_electronics (K=2) | docs=372
[Within-theme] amazon::sustainability_environment (K=2) | docs=279
[Within-theme] city::policy_regulation (K=6) | docs=625,335
[Within-theme] city::consumer_tradeoffs (K=6) | docs=187,361
[Within-theme] city::sustainability_environment (K=6) | docs=57,196
[Within-theme] city::waste_systems (K=6) | docs=20,352
[Within-theme] city::labels_trust_transparency (K=6) | docs=18,724
[Within-theme] city::circular_design (K=6) | docs=13,913
[Within-theme] city::ewaste_electronics (K=6) | docs=9,756
[Within-theme] city::ecommerce_logistics (K=6) | docs=8,692
[Within-theme] cit

In [1]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
IN_XLSX = Path("outputs") / "sustainability_public_opinion_within_theme_and_cooc.xlsx"
OUT_XLSX = Path("outputs") / "topic_names_subtopic.xlsx"

SHEET_SUBTOPIC = "within_theme_subtopic_summary"
SHEET_WINDOWS  = "windows_enriched"

# To avoid Excel cell-size issues, cap how many characters you keep per example window_text
MAX_CHARS_PER_EXAMPLE = 1000   # increase if you want, but Excel cells max at ~32,767 chars

# ------------------------------------------------------------
# LOAD: subtopic summary (small)
# ------------------------------------------------------------
subtopic = pd.read_excel(
    IN_XLSX,
    sheet_name=SHEET_SUBTOPIC,
    engine="openpyxl"
)

required_sub_cols = [
    "source", "dominant_theme", "subtopic_id", "subtopic_n",
    "subtopic_pct_within_theme", "subtopic_keywords_top12", "k_used", "docs_used"
]
missing = [c for c in required_sub_cols if c not in subtopic.columns]
if missing:
    raise ValueError(f"Missing columns in {SHEET_SUBTOPIC}: {missing}")

# normalize keys
subtopic["source"] = subtopic["source"].astype(str).str.lower().str.strip()
subtopic["dominant_theme"] = subtopic["dominant_theme"].astype(str).str.lower().str.strip()
subtopic["subtopic_id"] = pd.to_numeric(subtopic["subtopic_id"], errors="coerce")

# ------------------------------------------------------------
# LOAD: windows (can be large) -> ONLY needed columns
# ------------------------------------------------------------
windows = pd.read_excel(
    IN_XLSX,
    sheet_name=SHEET_WINDOWS,
    engine="openpyxl",
    usecols=["source", "dominant_theme", "subtopic_id", "subtopic_score", "window_text"]
)

required_win_cols = ["source", "dominant_theme", "subtopic_id", "subtopic_score", "window_text"]
missing = [c for c in required_win_cols if c not in windows.columns]
if missing:
    raise ValueError(f"Missing columns in {SHEET_WINDOWS}: {missing}")

# normalize keys
windows["source"] = windows["source"].astype(str).str.lower().str.strip()
windows["dominant_theme"] = windows["dominant_theme"].astype(str).str.lower().str.strip()
windows["subtopic_id"] = pd.to_numeric(windows["subtopic_id"], errors="coerce")
windows["subtopic_score"] = pd.to_numeric(windows["subtopic_score"], errors="coerce")

# Keep only rows that actually have subtopics assigned
windows = windows.dropna(subset=["subtopic_id", "subtopic_score", "window_text"]).copy()

# ------------------------------------------------------------
# BUILD: Top 5 highest subtopic_score examples per (source, theme, subtopic_id)
# ------------------------------------------------------------
# Sort once, then take top 5 rows per group efficiently
windows_sorted = windows.sort_values("subtopic_score", ascending=False)

top5_rows = (
    windows_sorted
    .groupby(["source", "dominant_theme", "subtopic_id"], as_index=False, sort=False)
    .head(5)
)

def pack_examples(df_group: pd.DataFrame) -> str:
    # df_group already in descending subtopic_score order due to global sort + head(5)
    lines = []
    for i, row in enumerate(df_group.itertuples(index=False), start=1):
        txt = str(row.window_text)
        if MAX_CHARS_PER_EXAMPLE is not None and len(txt) > MAX_CHARS_PER_EXAMPLE:
            txt = txt[:MAX_CHARS_PER_EXAMPLE] + " …"
        lines.append(f"[{i}] score={row.subtopic_score:.4f}\n{txt}")
    return "\n\n-----\n\n".join(lines)

top5_examples = (
    top5_rows
    .groupby(["source", "dominant_theme", "subtopic_id"], as_index=False)
    .apply(lambda g: pack_examples(g), include_groups=False)
    .rename(columns={None: "top5_highest_subtopic_score_window_text"})
)

# ------------------------------------------------------------
# MERGE: summary + examples
# ------------------------------------------------------------
out = subtopic.merge(
    top5_examples,
    on=["source", "dominant_theme", "subtopic_id"],
    how="left"
)

# Optional: add a blank column for you to manually name subtopics
out.insert(
    3,  # after source, dominant_theme, subtopic_id
    "subtopic_name",
    ""
)

# Keep exactly what you requested (plus subtopic_name, which you can remove if you don’t want it)
final_cols = [
    "source",
    "dominant_theme",
    "subtopic_id",
    "subtopic_name",  # optional helper column
    "subtopic_n",
    "subtopic_pct_within_theme",
    "subtopic_keywords_top12",
    "k_used",
    "docs_used",
    "top5_highest_subtopic_score_window_text"
]
out = out[final_cols]

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
OUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    out.to_excel(writer, index=False, sheet_name="topic_names_subtopic")

print(f"Saved: {OUT_XLSX.resolve()}")
print(f"Rows: {len(out):,}")

Saved: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\topic_names_subtopic.xlsx
Rows: 76


In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# CONFIG
# ----------------------------
IN_XLSX = Path("outputs") / "sustainability_public_opinion_within_theme_and_cooc.xlsx"
SHEET_WINDOWS = "windows_enriched"
SHEET_SUBTOPIC = "within_theme_subtopic_summary"

OUT_XLSX = Path("outputs") / "subtopic_comparative_results_named.xlsx"
FIG_DIR = Path("outputs") / "figures_subtopic_results_named"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TOP_THEMES = [
    "policy_regulation", "consumer_tradeoffs", "packaging_single_use",
    "ewaste_electronics", "labels_trust_transparency", "circular_design",
    "ecommerce_logistics", "sustainability_environment", "waste_systems"
]
TOP_SUBTOPICS_PER_THEME = 8  # for plots

# ----------------------------
# Helpers
# ----------------------------
def norm_header(h: str) -> str:
    """Normalize headers to match 'Subtopic Name' vs 'subtopic_name' etc."""
    return re.sub(r"[^a-z0-9]+", "", str(h).strip().lower())

def find_col_fuzzy(df: pd.DataFrame, candidates: list) -> str | None:
    cols = list(df.columns)
    norm_map = {norm_header(c): c for c in cols}
    for cand in candidates:
        key = norm_header(cand)
        if key in norm_map:
            return norm_map[key]
    return None

def safe_filename(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s))[:120]

def pretty_label(s: str, max_len: int = 70) -> str:
    """Make axis labels readable (Excel names can be long)."""
    s = str(s)
    return (s[:max_len] + "…") if len(s) > max_len else s

# ----------------------------
# LOAD
# ----------------------------
if not IN_XLSX.exists():
    raise FileNotFoundError(f"Missing input file: {IN_XLSX.resolve()}")

windows = pd.read_excel(IN_XLSX, sheet_name=SHEET_WINDOWS, engine="openpyxl")
subsum = pd.read_excel(IN_XLSX, sheet_name=SHEET_SUBTOPIC, engine="openpyxl")

# Normalize join keys
for df in (windows, subsum):
    df["source"] = df["source"].astype(str).str.lower().str.strip()
    df["dominant_theme"] = df["dominant_theme"].astype(str).str.lower().str.strip()
    df["subtopic_id"] = pd.to_numeric(df["subtopic_id"], errors="coerce")

# ----------------------------
# FORCE subtopic name mapping from within_theme_subtopic_summary
# ----------------------------
# Your column is "Subtopic Name" (with a space), so we detect it robustly:
name_col = find_col_fuzzy(subsum, ["Subtopic Name", "subtopic_name", "topic_name", "Subtopic_Name"])
if name_col is None:
    raise ValueError(
        "Could not find the subtopic name column in within_theme_subtopic_summary.\n"
        "Expected something like: 'Subtopic Name' or 'subtopic_name'."
    )

# Build mapping (unique)
map_df = subsum[["source", "dominant_theme", "subtopic_id", name_col]].copy()
map_df[name_col] = map_df[name_col].astype(str).str.strip()
map_df = map_df.drop_duplicates(subset=["source", "dominant_theme", "subtopic_id"])

# Merge and overwrite any existing name column
windows = windows.merge(
    map_df,
    on=["source", "dominant_theme", "subtopic_id"],
    how="left"
)

# Standardize final name column
windows["subtopic_name"] = windows[name_col]
windows = windows.drop(columns=[name_col], errors="ignore")

# ----------------------------
# Required columns for analysis
# ----------------------------
stance_col = find_col_fuzzy(windows, ["stance"])
sent_col = find_col_fuzzy(windows, ["sentiment_compound"])
emo_cols = [c for c in windows.columns if isinstance(c, str) and c.startswith("emo_")]

if stance_col is None:
    raise ValueError("Missing 'stance' column in windows_enriched.")
if sent_col is None:
    raise ValueError("Missing 'sentiment_compound' column in windows_enriched.")

# Only windows that received subtopic assignment
w = windows.dropna(subset=["subtopic_id"]).copy()
w["subtopic_name"] = w["subtopic_name"].fillna(
    w["dominant_theme"].astype(str) + "_subtopic_" + w["subtopic_id"].astype(int).astype(str)
)

# ----------------------------
# 1) Subtopic prevalence
# ----------------------------
subtopic_prevalence = (
    w.groupby(["source", "dominant_theme", "subtopic_id", "subtopic_name"])
     .size().reset_index(name="n")
)
subtopic_prevalence["pct_within_theme"] = subtopic_prevalence.groupby(
    ["source", "dominant_theme"]
)["n"].transform(lambda x: x / x.sum())
subtopic_prevalence["pct_within_source"] = subtopic_prevalence.groupby(
    ["source"]
)["n"].transform(lambda x: x / x.sum())

# ----------------------------
# 2) Stance-by-subtopic
# ----------------------------
stance_by_subtopic = (
    w.groupby(["source", "dominant_theme", "subtopic_id", "subtopic_name", w[stance_col]])
     .size().reset_index(name="n")
     .rename(columns={stance_col: "stance"})
)
stance_by_subtopic["pct_within_subtopic"] = stance_by_subtopic.groupby(
    ["source", "dominant_theme", "subtopic_id"]
)["n"].transform(lambda x: x / x.sum())

# ----------------------------
# 3) Sentiment-by-subtopic
# ----------------------------
sentiment_by_subtopic = (
    w.groupby(["source", "dominant_theme", "subtopic_id", "subtopic_name"])[sent_col]
     .agg(mean="mean", median="median", n="count")
     .reset_index()
)

# ----------------------------
# 4) Emotion-by-subtopic (optional)
# ----------------------------
emotion_by_subtopic = None
if len(emo_cols) > 0:
    emotion_by_subtopic = (
        w.groupby(["source", "dominant_theme", "subtopic_id", "subtopic_name"])[emo_cols]
         .mean(numeric_only=True)
         .reset_index()
    )

# ----------------------------
# 5) City vs Amazon comparison within each theme (top subtopics by total volume)
# ----------------------------
theme_compare_rows = []
for theme in TOP_THEMES:
    tmp = subtopic_prevalence[subtopic_prevalence["dominant_theme"] == theme].copy()
    if tmp.empty:
        continue

    tmp_total = tmp.groupby(["dominant_theme", "subtopic_name"])["n"].sum().reset_index()
    top_names = tmp_total.sort_values("n", ascending=False).head(TOP_SUBTOPICS_PER_THEME)["subtopic_name"].tolist()

    tmp2 = tmp[tmp["subtopic_name"].isin(top_names)]
    pv = tmp2.pivot_table(index="subtopic_name", columns="source", values="pct_within_theme", aggfunc="sum").fillna(0)
    pv = pv.reset_index()
    pv.insert(0, "dominant_theme", theme)
    theme_compare_rows.append(pv)

theme_comparison = pd.concat(theme_compare_rows, ignore_index=True) if theme_compare_rows else pd.DataFrame()

# ----------------------------
# SAVE tables
# ----------------------------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    windows.to_excel(writer, index=False, sheet_name="windows_named")
    subtopic_prevalence.to_excel(writer, index=False, sheet_name="subtopic_prevalence")
    stance_by_subtopic.to_excel(writer, index=False, sheet_name="stance_by_subtopic")
    sentiment_by_subtopic.to_excel(writer, index=False, sheet_name="sentiment_by_subtopic")
    if emotion_by_subtopic is not None:
        emotion_by_subtopic.to_excel(writer, index=False, sheet_name="emotion_by_subtopic")
    if not theme_comparison.empty:
        theme_comparison.to_excel(writer, index=False, sheet_name="theme_comparison_city_vs_amz")

print(f"Saved summary workbook: {OUT_XLSX.resolve()}")

# ----------------------------
# PLOTS
# ----------------------------
stance_order = ["support", "conditional", "oppose", "neutral"]

# A) City vs Amazon subtopic prevalence within each theme
if not theme_comparison.empty:
    for theme in theme_comparison["dominant_theme"].unique():
        sub = theme_comparison[theme_comparison["dominant_theme"] == theme].copy()
        if sub.empty:
            continue
        for col in ["city", "amazon"]:
            if col not in sub.columns:
                sub[col] = 0.0

        labels = [pretty_label(x) for x in sub["subtopic_name"].tolist()]
        x = np.arange(len(labels))
        width = 0.38

        plt.figure(figsize=(12, max(4, 0.35 * len(labels) + 2)))
        plt.barh(x - width/2, sub["city"], height=0.35, label="city")
        plt.barh(x + width/2, sub["amazon"], height=0.35, label="amazon")
        plt.yticks(x, labels)
        plt.xlabel("Proportion within theme")
        plt.title(f"City vs Amazon: subtopic prevalence within theme = {theme}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"compare_subtopic_prevalence_{safe_filename(theme)}.png", dpi=220)
        plt.close()

# B) Stance distribution by subtopic (top subtopics per theme, per source)
for src in ["city", "amazon"]:
    for theme in TOP_THEMES:
        top = subtopic_prevalence[(subtopic_prevalence["source"] == src) &
                                  (subtopic_prevalence["dominant_theme"] == theme)] \
                .sort_values("n", ascending=False).head(TOP_SUBTOPICS_PER_THEME)
        if top.empty:
            continue

        sub_ids = top["subtopic_id"].tolist()
        sb = stance_by_subtopic[(stance_by_subtopic["source"] == src) &
                                (stance_by_subtopic["dominant_theme"] == theme) &
                                (stance_by_subtopic["subtopic_id"].isin(sub_ids))].copy()
        if sb.empty:
            continue

        pivot = sb.pivot_table(index="subtopic_name", columns="stance", values="pct_within_subtopic", aggfunc="sum").fillna(0.0)
        for s in stance_order:
            if s not in pivot.columns:
                pivot[s] = 0.0
        pivot = pivot[stance_order]

        ylabels = [pretty_label(x) for x in pivot.index.tolist()]
        y = np.arange(len(ylabels))
        left = np.zeros(len(ylabels))

        plt.figure(figsize=(12, max(4, 0.35 * len(ylabels) + 2)))
        for st in stance_order:
            plt.barh(y, pivot[st].values, left=left, height=0.7, label=st)
            left += pivot[st].values

        plt.yticks(y, ylabels)
        plt.xlabel("Proportion within subtopic")
        plt.title(f"{src}: stance distribution by subtopic (theme={theme}, top {TOP_SUBTOPICS_PER_THEME})")
        plt.legend(ncol=4, bbox_to_anchor=(0.5, -0.08), loc="upper center")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"stance_by_subtopic_{src}_{safe_filename(theme)}.png", dpi=220)
        plt.close()

# C) Sentiment heatmaps per theme (mean sentiment, city vs amazon)
for theme in TOP_THEMES:
    sub = sentiment_by_subtopic[sentiment_by_subtopic["dominant_theme"] == theme].copy()
    if sub.empty:
        continue

    total = sub.groupby("subtopic_name")["n"].sum().reset_index().sort_values("n", ascending=False)
    top_names = total.head(TOP_SUBTOPICS_PER_THEME)["subtopic_name"].tolist()
    sub = sub[sub["subtopic_name"].isin(top_names)]

    pv = sub.pivot_table(index="subtopic_name", columns="source", values="mean", aggfunc="mean").fillna(0.0)
    if pv.empty:
        continue

    plt.figure(figsize=(6, max(4, 0.35 * len(pv.index) + 2)))
    plt.imshow(pv.values, aspect="auto")
    plt.yticks(range(len(pv.index)), [pretty_label(x) for x in pv.index.tolist()])
    plt.xticks(range(len(pv.columns)), pv.columns.tolist())
    plt.colorbar(label="Mean sentiment (compound)")
    plt.title(f"Mean sentiment by subtopic (theme={theme})")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"sentiment_heatmap_{safe_filename(theme)}.png", dpi=220)
    plt.close()

# D) Emotion heatmaps (if emotion columns exist)
if emotion_by_subtopic is not None and len(emo_cols) > 0:
    key_emotions = [c for c in ["emo_trust", "emo_anger", "emo_fear", "emo_joy", "emo_sadness"] if c in emo_cols]
    if not key_emotions:
        key_emotions = emo_cols[:6]

    for theme in TOP_THEMES:
        sub = emotion_by_subtopic[emotion_by_subtopic["dominant_theme"] == theme].copy()
        if sub.empty:
            continue

        prev = subtopic_prevalence[subtopic_prevalence["dominant_theme"] == theme].copy()
        if prev.empty:
            continue

        top_names = prev.groupby("subtopic_name")["n"].sum().sort_values(ascending=False).head(TOP_SUBTOPICS_PER_THEME).index.tolist()
        sub = sub[sub["subtopic_name"].isin(top_names)]

        for src in ["city", "amazon"]:
            ssub = sub[sub["source"] == src].copy()
            if ssub.empty:
                continue
            mat = ssub.set_index("subtopic_name")[key_emotions].fillna(0.0)

            plt.figure(figsize=(10, max(4, 0.35 * len(mat.index) + 2)))
            plt.imshow(mat.values, aspect="auto")
            plt.yticks(range(len(mat.index)), [pretty_label(x) for x in mat.index.tolist()])
            plt.xticks(range(len(mat.columns)), mat.columns.tolist(), rotation=45, ha="right")
            plt.colorbar(label="Mean emotion score")
            plt.title(f"{src}: emotion profile by subtopic (theme={theme})")
            plt.tight_layout()
            plt.savefig(FIG_DIR / f"emotion_heatmap_{src}_{safe_filename(theme)}.png", dpi=220)
            plt.close()

print(f"Saved plots to: {FIG_DIR.resolve()}")

In [2]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIG — EDIT THESE TWO IF NEEDED
# ============================================================
INPUT_XLSX = Path("Final results.xlsx")     # <-- your file name
SHEET_NAME = "results"               # <-- your sheet name (will auto-detect if not found)

OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "final_figures"
OUT_TABLES = OUT_DIR / "final_results_tables.xlsx"
OUT_MD = OUT_DIR / "draft_manuscript.md"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Which themes to feature in the written Results section by default
FEATURE_THEMES = ["policy_regulation", "packaging_single_use", "consumer_tradeoffs"]

TOP_SUBTOPICS_PER_THEME = 8
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ============================================================
# Helpers
# ============================================================
def norm_header(h: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(h).strip().lower())

def find_col(df: pd.DataFrame, candidates: list) -> str | None:
    cols = list(df.columns)
    m = {norm_header(c): c for c in cols}
    for cand in candidates:
        k = norm_header(cand)
        if k in m:
            return m[k]
    return None

def safe_filename(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s))[:120]

def pct(x):
    return f"{100*x:.1f}%"

# ============================================================
# Load workbook + sheet
# ============================================================
if not INPUT_XLSX.exists():
    raise FileNotFoundError(f"Missing input file: {INPUT_XLSX.resolve()}")

xls = pd.ExcelFile(INPUT_XLSX, engine="openpyxl")
sheets = xls.sheet_names

if SHEET_NAME not in sheets:
    # auto-detect: pick the first sheet that contains "final"
    candidates = [s for s in sheets if "final" in s.lower()]
    if candidates:
        SHEET_NAME = candidates[0]
    else:
        SHEET_NAME = sheets[0]

df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME, engine="openpyxl")

# ============================================================
# Required columns (fuzzy matching)
# ============================================================
col_source  = find_col(df, ["source"])
col_theme   = find_col(df, ["dominant_theme", "theme"])
col_subid   = find_col(df, ["subtopic_id", "subtopicid"])
col_subname = find_col(df, ["Subtopic Name", "subtopic_name", "topic_name"])
col_stance  = find_col(df, ["stance"])
col_sent    = find_col(df, ["sentiment_compound", "sentiment"])

missing = []
for name, col in [
    ("source", col_source),
    ("dominant_theme", col_theme),
    ("subtopic_id", col_subid),
    ("Subtopic Name", col_subname),
    ("stance", col_stance),
    ("sentiment_compound", col_sent),
]:
    if col is None:
        missing.append(name)

if missing:
    raise ValueError(
        f"Missing required columns in sheet '{SHEET_NAME}': {missing}\n"
        f"Available columns are:\n{list(df.columns)}"
    )

# Normalize / clean
df[col_source] = df[col_source].astype(str).str.lower().str.strip()
df[col_theme]  = df[col_theme].astype(str).str.lower().str.strip()
df[col_subid]  = pd.to_numeric(df[col_subid], errors="coerce")
df[col_subname] = df[col_subname].astype(str).str.strip()
df[col_sent]   = pd.to_numeric(df[col_sent], errors="coerce")
df[col_stance] = df[col_stance].astype(str).str.lower().str.strip()

# Optional columns
emo_cols = [c for c in df.columns if isinstance(c, str) and c.startswith("emo_")]
col_year = find_col(df, ["year"])
col_state = find_col(df, ["state_name", "state"])

# Only use rows that have subtopics assigned
w = df.dropna(subset=[col_subid]).copy()

# ============================================================
# 1) Overall stance distribution by source
# ============================================================
stance_dist = (
    w.groupby([col_source, col_stance]).size().reset_index(name="n")
)
stance_dist["pct"] = stance_dist.groupby(col_source)["n"].transform(lambda x: x / x.sum())

# ============================================================
# 2) Subtopic prevalence
# ============================================================
sub_prev = (
    w.groupby([col_source, col_theme, col_subid, col_subname]).size().reset_index(name="n")
)
sub_prev["pct_within_theme"] = sub_prev.groupby([col_source, col_theme])["n"].transform(lambda x: x / x.sum())
sub_prev["pct_within_source"] = sub_prev.groupby([col_source])["n"].transform(lambda x: x / x.sum())

# ============================================================
# 3) Stance by subtopic
# ============================================================
stance_by_sub = (
    w.groupby([col_source, col_theme, col_subid, col_subname, col_stance]).size().reset_index(name="n")
)
stance_by_sub["pct_within_subtopic"] = stance_by_sub.groupby([col_source, col_theme, col_subid])["n"].transform(lambda x: x / x.sum())

# ============================================================
# 4) Sentiment by subtopic
# ============================================================
sent_by_sub = (
    w.groupby([col_source, col_theme, col_subid, col_subname])[col_sent]
     .agg(mean="mean", median="median", n="count")
     .reset_index()
)

# ============================================================
# 5) Emotion by subtopic (optional)
# ============================================================
emo_by_sub = None
if emo_cols:
    emo_by_sub = (
        w.groupby([col_source, col_theme, col_subid, col_subname])[emo_cols]
         .mean(numeric_only=True)
         .reset_index()
    )

# ============================================================
# 6) City vs Amazon mismatch within each theme
# ============================================================
pv = sub_prev.pivot_table(
    index=[col_theme, col_subname],
    columns=col_source,
    values="pct_within_theme",
    aggfunc="sum"
).fillna(0.0)

for src in ["city", "amazon"]:
    if src not in pv.columns:
        pv[src] = 0.0

pv["diff_city_minus_amazon"] = pv["city"] - pv["amazon"]
mismatch = pv.reset_index().sort_values("diff_city_minus_amazon", ascending=False)

# ============================================================
# 7) Extra “writing-friendly” tables
#    - Top opposed/conditional subtopics per source
# ============================================================
# build oppose+conditional share per subtopic
stance_piv = stance_by_sub.pivot_table(
    index=[col_source, col_theme, col_subid, col_subname],
    columns=col_stance,
    values="pct_within_subtopic",
    aggfunc="sum"
).fillna(0.0)

for s in ["oppose", "conditional"]:
    if s not in stance_piv.columns:
        stance_piv[s] = 0.0

stance_piv["oppose_plus_conditional"] = stance_piv["oppose"] + stance_piv["conditional"]
top_resistance = stance_piv.reset_index().sort_values("oppose_plus_conditional", ascending=False)

# ============================================================
# SAVE TABLES
# ============================================================
with pd.ExcelWriter(OUT_TABLES, engine="openpyxl") as writer:
    stance_dist.to_excel(writer, index=False, sheet_name="stance_distribution")
    sub_prev.to_excel(writer, index=False, sheet_name="subtopic_prevalence")
    stance_by_sub.to_excel(writer, index=False, sheet_name="stance_by_subtopic")
    sent_by_sub.to_excel(writer, index=False, sheet_name="sentiment_by_subtopic")
    mismatch.to_excel(writer, index=False, sheet_name="city_vs_amazon_mismatch")
    top_resistance.to_excel(writer, index=False, sheet_name="top_resistance_subtopics")
    if emo_by_sub is not None:
        emo_by_sub.to_excel(writer, index=False, sheet_name="emotion_by_subtopic")

print(f"Saved tables: {OUT_TABLES.resolve()}")

# ============================================================
# FIGURES
# ============================================================
# Figure 1: stance distribution
fig1 = FIG_DIR / "fig1_stance_distribution_city_vs_amazon.png"
pivot_st = stance_dist.pivot(index=col_stance, columns=col_source, values="pct").fillna(0.0)
order = ["support", "conditional", "oppose", "neutral"]
pivot_st = pivot_st.reindex([o for o in order if o in pivot_st.index])
ax = pivot_st.plot(kind="bar", figsize=(7,4))
ax.set_title("Stance distribution by corpus")
ax.set_ylabel("Proportion of windows")
plt.tight_layout()
plt.savefig(fig1, dpi=220)
plt.close()

# Figure 2+: mismatch plots for featured themes
for theme in FEATURE_THEMES:
    subm = mismatch[mismatch[col_theme] == theme].copy()
    if subm.empty:
        continue

    top_city = subm.head(6)  # city-heavy
    top_amz = subm.tail(6).sort_values("diff_city_minus_amazon")  # amazon-heavy
    show = pd.concat([top_city, top_amz], ignore_index=True)

    fig = FIG_DIR / f"fig_mismatch_{safe_filename(theme)}.png"
    plt.figure(figsize=(10, max(4, 0.35 * len(show) + 2)))
    y = np.arange(len(show))
    plt.barh(y, show["diff_city_minus_amazon"])
    plt.yticks(y, show[col_subname].astype(str).tolist())
    plt.axvline(0, linewidth=1)
    plt.title(f"City–Amazon mismatch (pct within theme): {theme}")
    plt.xlabel("City pct − Amazon pct")
    plt.tight_layout()
    plt.savefig(fig, dpi=220)
    plt.close()

print(f"Saved figures to: {FIG_DIR.resolve()}")

# ============================================================
# DRAFT WRITING SKELETON (Markdown, auto-filled)
# ============================================================
def stance_summary_for(src: str) -> str:
    sub = stance_dist[stance_dist[col_source] == src].copy()
    d = {r[col_stance]: r["pct"] for _, r in sub.iterrows()}
    return ", ".join([f"{k}={pct(d.get(k,0))}" for k in ["support","conditional","oppose","neutral"]])

n_city = int((w[col_source] == "city").sum())
n_amz = int((w[col_source] == "amazon").sum())

md = []
md.append("# Draft Paper Skeleton (auto-filled from Final Results)\n")

md.append("## Suggested Title\n")
md.append("From City Hall to Checkout: Comparing Attitudes Toward Sustainable Policy in Municipal Transcripts and Amazon Reviews\n\n")

md.append("## Abstract (draft)\n")
md.append(f"- Window-level dataset used for subtopic analysis: city n={n_city:,}, amazon n={n_amz:,}.\n")
md.append(f"- Stance distribution (city): {stance_summary_for('city')}.\n")
md.append(f"- Stance distribution (amazon): {stance_summary_for('amazon')}.\n")
md.append("- Key finding preview: large mismatches appear within major themes (policy_regulation, packaging, consumer_tradeoffs).\n\n")

md.append("## 1. Introduction\n")
md.append("- Motivation: sustainable policy is expanding; acceptance/resistance varies.\n")
md.append("- Gap: few studies compare civic discourse vs marketplace experience using comparable text units.\n")
md.append("- Contribution: keyword-window extraction + theme + attitudes + within-theme subtopics; compare city vs Amazon.\n\n")

md.append("## 2. Conceptual Background\n")
md.append("- Define attitudes: stance (support/oppose/conditional/neutral) + sentiment (+ emotion optional).\n")
md.append("- City discourse reflects governance/policy framing; Amazon reflects consumer experience and tradeoffs.\n\n")

md.append("## 3. Data\n")
md.append("- City: council transcript windows.\n")
md.append("- Amazon: review windows.\n")
md.append("- Unit: keyword-centered window (±400 chars).\n\n")

md.append("## 4. Methods\n")
md.append("- Theme assignment: dictionary-based categories (policy, packaging, e-waste, trust, tradeoffs).\n")
md.append("- Attitude measures: stance classification + sentiment.\n")
md.append("- Topic discovery: within-theme subtopic modeling (LDA per theme) to avoid noisy global topics.\n\n")

md.append("## 5. Results\n")
md.append("### 5.1 Overall attitudes\n")
md.append(f"- City stance: {stance_summary_for('city')}.\n")
md.append(f"- Amazon stance: {stance_summary_for('amazon')}.\n\n")

md.append("### 5.2 Civic vs marketplace mismatch within themes\n")
for theme in FEATURE_THEMES:
    subm = mismatch[mismatch[col_theme] == theme].copy()
    if subm.empty:
        continue
    city_heavy = subm.head(3)[[col_subname, "diff_city_minus_amazon"]].values.tolist()
    amz_heavy = subm.tail(3).sort_values("diff_city_minus_amazon")[[col_subname, "diff_city_minus_amazon"]].values.tolist()

    md.append(f"**Theme: {theme}**\n")
    md.append("- City-heavy subtopics (City pct − Amazon pct):\n")
    for name, diffv in city_heavy:
        md.append(f"  - {name}: {diffv:+.3f}\n")
    md.append("- Amazon-heavy subtopics (City pct − Amazon pct):\n")
    for name, diffv in amz_heavy:
        md.append(f"  - {name}: {diffv:+.3f}\n")
    md.append("\n")

md.append("### 5.3 Drivers of resistance\n")
md.append("- Use table `top_resistance_subtopics` to describe which subtopics have highest oppose+conditional shares.\n\n")

md.append("## 6. Discussion\n")
md.append("- Interpret mismatches: institutional framing vs lived consumer constraints.\n")
md.append("- Policy implications: communication, feasibility/equity, enforcement credibility.\n")
md.append("- Platform/industry implications: trust/verification, packaging, repairability cues.\n\n")

md.append("## Figures & Tables\n")
md.append(f"- Figures saved in: {FIG_DIR}\n")
md.append(f"- Summary tables saved in: {OUT_TABLES.name}\n")

OUT_MD.write_text("\n".join(md), encoding="utf-8")
print(f"Saved writing skeleton: {OUT_MD.resolve()}")

Saved tables: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\final_results_tables.xlsx
Saved figures to: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\final_figures
Saved writing skeleton: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\draft_manuscript.md


In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
INPUT_XLSX = Path("Final results.xlsx")
SHEET_PREFERENCE = ["Final Results", "results ", "results"]  # tries these first

OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "spc_figures"
OUT_TABLES = OUT_DIR / "spc_results_tables.xlsx"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FOCUS_THEMES = ["policy_regulation", "packaging_single_use", "consumer_tradeoffs"]
TOP_SUBTOPICS_PER_THEME = 8
TOP_RESISTANCE_N = 15

# =========================================================
# Helpers
# =========================================================
def norm_header(h: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(h).strip().lower())

def find_col(df: pd.DataFrame, candidates: list) -> str | None:
    cols = list(df.columns)
    m = {norm_header(c): c for c in cols}
    for cand in candidates:
        k = norm_header(cand)
        if k in m:
            return m[k]
    return None

def safe_filename(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s))[:120]

def pct(x: float) -> float:
    return float(x) if pd.notna(x) else 0.0

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

# =========================================================
# Load sheet
# =========================================================
if not INPUT_XLSX.exists():
    raise FileNotFoundError(f"Missing file: {INPUT_XLSX.resolve()}")

xls = pd.ExcelFile(INPUT_XLSX, engine="openpyxl")
sheet = None
for s in SHEET_PREFERENCE:
    if s in xls.sheet_names:
        sheet = s
        break
if sheet is None:
    sheet = xls.sheet_names[0]

df = pd.read_excel(INPUT_XLSX, sheet_name=sheet, engine="openpyxl")

# =========================================================
# Required columns (fuzzy match)
# =========================================================
col_source  = find_col(df, ["source"])
col_theme   = find_col(df, ["dominant_theme", "theme"])
col_subid   = find_col(df, ["subtopic_id", "subtopicid"])
col_subname = find_col(df, ["Subtopic Name", "subtopic_name", "topic_name", "subtopicname"])
col_stance  = find_col(df, ["stance"])
col_sent    = find_col(df, ["sentiment_compound", "sentiment", "sentimentcompound"])

required_missing = []
for name, col in [
    ("source", col_source),
    ("dominant_theme", col_theme),
    ("subtopic_id", col_subid),
    ("subtopic_name / Subtopic Name", col_subname),
    ("stance", col_stance),
    ("sentiment_compound", col_sent),
]:
    if col is None:
        required_missing.append(name)

if required_missing:
    raise ValueError(
        f"Missing required columns in sheet '{sheet}': {required_missing}\n"
        f"Available columns:\n{list(df.columns)}"
    )

# Optional columns
emo_cols = [c for c in df.columns if isinstance(c, str) and c.startswith("emo_")]
col_year = find_col(df, ["year"])
col_state = find_col(df, ["state_name", "state"])

# Normalize key fields
df[col_source] = df[col_source].astype(str).str.lower().str.strip()
df[col_theme] = df[col_theme].astype(str).str.lower().str.strip()
df[col_stance] = df[col_stance].astype(str).str.lower().str.strip()

df[col_subid] = pd.to_numeric(df[col_subid], errors="coerce")
df[col_sent] = pd.to_numeric(df[col_sent], errors="coerce")
df[col_subname] = df[col_subname].astype(str).str.strip()

# Keep only rows with subtopic assignment for subtopic analyses
w = df.dropna(subset=[col_subid]).copy()

# =========================================================
# TABLE 2: Corpus summary
# =========================================================
corpus_summary = []
for src in sorted(w[col_source].unique()):
    sub = w[w[col_source] == src]
    entry = {
        "source": src,
        "windows_n": int(len(sub)),
        "unique_themes_n": int(sub[col_theme].nunique()),
        "unique_subtopics_n": int(sub[[col_theme, col_subid]].drop_duplicates().shape[0]),
        "mean_sentiment": float(sub[col_sent].mean()),
        "median_sentiment": float(sub[col_sent].median())
    }
    if col_state and src == "city":
        entry["unique_states_n"] = int(sub[col_state].nunique())
    if col_year and src == "city":
        entry["year_min"] = int(pd.to_numeric(sub[col_year], errors="coerce").min())
        entry["year_max"] = int(pd.to_numeric(sub[col_year], errors="coerce").max())
    corpus_summary.append(entry)

corpus_summary = pd.DataFrame(corpus_summary)

# =========================================================
# THEME prevalence table
# =========================================================
theme_prev = (
    w.groupby([col_source, col_theme]).size().reset_index(name="n")
)
theme_prev["pct_within_source"] = theme_prev.groupby(col_source)["n"].transform(lambda x: x / x.sum())

# =========================================================
# FIGURE 2: Theme prevalence (City vs Amazon)
# =========================================================
fig2 = FIG_DIR / "fig2_theme_prevalence_city_vs_amazon.png"
pivot_theme = theme_prev.pivot_table(index=col_theme, columns=col_source, values="pct_within_source", aggfunc="sum").fillna(0.0)
# sort by city share (or overall)
if "city" in pivot_theme.columns:
    pivot_theme = pivot_theme.sort_values("city", ascending=False)
ax = pivot_theme.plot(kind="bar", figsize=(10,5))
ax.set_title("Theme prevalence by corpus")
ax.set_ylabel("Proportion within corpus")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(fig2, dpi=220)
plt.close()

# =========================================================
# SUBTOPIC prevalence table
# =========================================================
sub_prev = (
    w.groupby([col_source, col_theme, col_subid, col_subname]).size().reset_index(name="n")
)
sub_prev["pct_within_theme"] = sub_prev.groupby([col_source, col_theme])["n"].transform(lambda x: x / x.sum())

# =========================================================
# TABLE 3: Subtopic summary for focus themes (top per source)
# =========================================================
subtopic_summary_rows = []
for theme in FOCUS_THEMES:
    for src in ["city", "amazon"]:
        sub = sub_prev[(sub_prev[col_theme] == theme) & (sub_prev[col_source] == src)].copy()
        if sub.empty:
            continue
        top = sub.sort_values("n", ascending=False).head(TOP_SUBTOPICS_PER_THEME)
        top = top.assign(rank_within_theme=np.arange(1, len(top)+1))
        subtopic_summary_rows.append(top)

subtopic_summary = pd.concat(subtopic_summary_rows, ignore_index=True) if subtopic_summary_rows else pd.DataFrame()

# =========================================================
# FIGURE 3: Subtopic prevalence mismatch within each focus theme
# =========================================================
for theme in FOCUS_THEMES:
    sub = sub_prev[sub_prev[col_theme] == theme].copy()
    if sub.empty:
        continue

    # choose top subtopic names by total n across corpora
    totals = sub.groupby(col_subname)["n"].sum().reset_index().sort_values("n", ascending=False)
    top_names = totals.head(TOP_SUBTOPICS_PER_THEME)[col_subname].tolist()
    sub = sub[sub[col_subname].isin(top_names)]

    pv = sub.pivot_table(index=col_subname, columns=col_source, values="pct_within_theme", aggfunc="sum").fillna(0.0)
    for s in ["city", "amazon"]:
        if s not in pv.columns:
            pv[s] = 0.0
    pv = pv.sort_values("city", ascending=False)

    fig3 = FIG_DIR / f"fig3_subtopic_prevalence_compare_{safe_filename(theme)}.png"
    y = np.arange(len(pv.index))
    plt.figure(figsize=(12, max(4, 0.35*len(pv.index)+2)))
    plt.barh(y - 0.18, pv["city"].values, height=0.35, label="city")
    plt.barh(y + 0.18, pv["amazon"].values, height=0.35, label="amazon")
    plt.yticks(y, pv.index.tolist())
    plt.xlabel("Proportion within theme")
    plt.title(f"Subtopic prevalence within theme: {theme} (City vs Amazon)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(fig3, dpi=220)
    plt.close()

# =========================================================
# STANCE distribution overall
# =========================================================
stance_dist = (
    w.groupby([col_source, col_stance]).size().reset_index(name="n")
)
stance_dist["pct"] = stance_dist.groupby(col_source)["n"].transform(lambda x: x / x.sum())

# =========================================================
# FIGURE 4: Overall stance distribution
# =========================================================
fig4 = FIG_DIR / "fig4_stance_distribution_city_vs_amazon.png"
pivot_stance = stance_dist.pivot_table(index=col_stance, columns=col_source, values="pct", aggfunc="sum").fillna(0.0)
order = ["support", "conditional", "oppose", "neutral"]
pivot_stance = pivot_stance.reindex([o for o in order if o in pivot_stance.index])
ax = pivot_stance.plot(kind="bar", figsize=(7,4))
ax.set_title("Overall stance distribution by corpus")
ax.set_ylabel("Proportion of windows")
plt.tight_layout()
plt.savefig(fig4, dpi=220)
plt.close()

# =========================================================
# STANCE by subtopic
# =========================================================
stance_by_sub = (
    w.groupby([col_source, col_theme, col_subid, col_subname, col_stance]).size().reset_index(name="n")
)
stance_by_sub["pct_within_subtopic"] = stance_by_sub.groupby([col_source, col_theme, col_subid])["n"].transform(lambda x: x / x.sum())

# =========================================================
# FIGURE 5: Stance-by-subtopic stacked bars (focus themes, per source)
# =========================================================
stance_order = ["support", "conditional", "oppose", "neutral"]
for src in ["city", "amazon"]:
    for theme in FOCUS_THEMES:
        top = sub_prev[(sub_prev[col_source] == src) & (sub_prev[col_theme] == theme)].sort_values("n", ascending=False).head(TOP_SUBTOPICS_PER_THEME)
        if top.empty:
            continue
        keep_names = top[col_subname].tolist()

        sb = stance_by_sub[(stance_by_sub[col_source] == src) & (stance_by_sub[col_theme] == theme) & (stance_by_sub[col_subname].isin(keep_names))].copy()
        if sb.empty:
            continue

        pv = sb.pivot_table(index=col_subname, columns=col_stance, values="pct_within_subtopic", aggfunc="sum").fillna(0.0)
        for st in stance_order:
            if st not in pv.columns:
                pv[st] = 0.0
        pv = pv[stance_order]

        fig5 = FIG_DIR / f"fig5_stance_by_subtopic_{src}_{safe_filename(theme)}.png"
        y = np.arange(len(pv.index))
        left = np.zeros(len(pv.index))
        plt.figure(figsize=(12, max(4, 0.35*len(pv.index)+2)))
        for st in stance_order:
            plt.barh(y, pv[st].values, left=left, height=0.7, label=st)
            left += pv[st].values
        plt.yticks(y, pv.index.tolist())
        plt.xlabel("Proportion within subtopic")
        plt.title(f"{src}: stance by subtopic (theme={theme})")
        plt.legend(ncol=4, bbox_to_anchor=(0.5, -0.08), loc="upper center")
        plt.tight_layout()
        plt.savefig(fig5, dpi=220)
        plt.close()

# =========================================================
# SENTIMENT by subtopic
# =========================================================
sent_by_sub = (
    w.groupby([col_source, col_theme, col_subid, col_subname])[col_sent]
     .agg(mean="mean", median="median", n="count")
     .reset_index()
)

# =========================================================
# FIGURE 6: Sentiment heatmaps (focus themes)
# rows=subtopic, cols=city/amazon
# =========================================================
for theme in FOCUS_THEMES:
    sub = sent_by_sub[sent_by_sub[col_theme] == theme].copy()
    if sub.empty:
        continue

    totals = sub.groupby(col_subname)["n"].sum().reset_index().sort_values("n", ascending=False)
    top_names = totals.head(TOP_SUBTOPICS_PER_THEME)[col_subname].tolist()
    sub = sub[sub[col_subname].isin(top_names)]

    pv = sub.pivot_table(index=col_subname, columns=col_source, values="mean", aggfunc="mean").fillna(0.0)
    if pv.empty:
        continue
    cols = list(pv.columns)

    fig6 = FIG_DIR / f"fig6_sentiment_heatmap_{safe_filename(theme)}.png"
    plt.figure(figsize=(6, max(4, 0.35*len(pv.index)+2)))
    plt.imshow(pv.values, aspect="auto")
    plt.yticks(range(len(pv.index)), pv.index.tolist())
    plt.xticks(range(len(cols)), cols)
    plt.colorbar(label="Mean sentiment (compound)")
    plt.title(f"Mean sentiment by subtopic (theme={theme})")
    plt.tight_layout()
    plt.savefig(fig6, dpi=220)
    plt.close()

# =========================================================
# FIGURE (optional): Emotion heatmaps for focus themes (if emo_ cols exist)
# =========================================================
emo_by_sub = None
if emo_cols:
    emo_by_sub = (
        w.groupby([col_source, col_theme, col_subid, col_subname])[emo_cols]
         .mean(numeric_only=True)
         .reset_index()
    )
    key_emotions = [c for c in ["emo_trust", "emo_anger", "emo_fear", "emo_joy", "emo_sadness"] if c in emo_cols]
    if not key_emotions:
        key_emotions = emo_cols[:6]

    for theme in FOCUS_THEMES:
        sub = emo_by_sub[emo_by_sub[col_theme] == theme].copy()
        if sub.empty:
            continue

        # top subtopics by total n across sources
        totals = sub_prev[sub_prev[col_theme] == theme].groupby(col_subname)["n"].sum().sort_values(ascending=False)
        top_names = totals.head(TOP_SUBTOPICS_PER_THEME).index.tolist()
        sub = sub[sub[col_subname].isin(top_names)]

        for src in ["city", "amazon"]:
            ssub = sub[sub[col_source] == src].copy()
            if ssub.empty:
                continue

            mat = ssub.set_index(col_subname)[key_emotions].fillna(0.0)
            figE = FIG_DIR / f"fig_emotion_heatmap_{src}_{safe_filename(theme)}.png"
            plt.figure(figsize=(10, max(4, 0.35*len(mat.index)+2)))
            plt.imshow(mat.values, aspect="auto")
            plt.yticks(range(len(mat.index)), mat.index.tolist())
            plt.xticks(range(len(mat.columns)), mat.columns.tolist(), rotation=45, ha="right")
            plt.colorbar(label="Mean emotion score")
            plt.title(f"{src}: emotion by subtopic (theme={theme})")
            plt.tight_layout()
            plt.savefig(figE, dpi=220)
            plt.close()

# =========================================================
# TABLE 5: Most resistance subtopics (oppose+conditional)
# =========================================================
stance_piv = stance_by_sub.pivot_table(
    index=[col_source, col_theme, col_subid, col_subname],
    columns=col_stance,
    values="pct_within_subtopic",
    aggfunc="sum"
).fillna(0.0)

for st in ["oppose", "conditional", "support", "neutral"]:
    if st not in stance_piv.columns:
        stance_piv[st] = 0.0

stance_piv["resistance_oppose_plus_conditional"] = stance_piv["oppose"] + stance_piv["conditional"]
resistance = stance_piv.reset_index().sort_values("resistance_oppose_plus_conditional", ascending=False)

# add raw window counts per subtopic
sizes = w.groupby([col_source, col_theme, col_subid, col_subname]).size().reset_index(name="subtopic_windows_n")
resistance = resistance.merge(sizes, on=[col_source, col_theme, col_subid, col_subname], how="left")

top_resistance_overall = (
    resistance.sort_values(["resistance_oppose_plus_conditional"], ascending=False)
    .groupby(col_source, as_index=False).head(TOP_RESISTANCE_N)
)

top_resistance_by_theme = (
    resistance[resistance[col_theme].isin(FOCUS_THEMES)]
    .sort_values(["resistance_oppose_plus_conditional"], ascending=False)
    .groupby([col_source, col_theme], as_index=False).head(TOP_RESISTANCE_N)
)

# =========================================================
# FIGURE 7: Resistance hotspots (top N per source)
# =========================================================
fig7 = FIG_DIR / "fig7_top_resistance_subtopics_by_source.png"
# For a clean plot: show top 10 overall per source as separate panels
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)
for i, src in enumerate(["city", "amazon"]):
    sub = top_resistance_overall[top_resistance_overall[col_source] == src].head(10).copy()
    if sub.empty:
        axes[i].set_axis_off()
        continue
    axes[i].barh(sub[col_subname].astype(str).tolist()[::-1],
                 sub["resistance_oppose_plus_conditional"].tolist()[::-1])
    axes[i].set_title(f"Top resistance subtopics: {src}")
    axes[i].set_xlabel("Oppose + Conditional share")
plt.tight_layout()
plt.savefig(fig7, dpi=220)
plt.close()

# =========================================================
# Optional city-only: state stance distribution (top 15 states)
# =========================================================
state_stance_table = None
if col_state and "city" in w[col_source].unique():
    city = w[w[col_source] == "city"].copy()
    top_states = city[col_state].astype(str).value_counts().head(15).index.tolist()
    city = city[city[col_state].astype(str).isin(top_states)]
    state_stance = (
        city.groupby([col_state, col_stance]).size().reset_index(name="n")
    )
    state_stance["pct_within_state"] = state_stance.groupby(col_state)["n"].transform(lambda x: x / x.sum())
    state_stance_table = state_stance

    # plot
    figS = FIG_DIR / "fig_city_state_stance_top15.png"
    pv = state_stance.pivot_table(index=col_state, columns=col_stance, values="pct_within_state", aggfunc="sum").fillna(0.0)
    for st in stance_order:
        if st not in pv.columns:
            pv[st] = 0.0
    pv = pv[stance_order]
    ax = pv.plot(kind="bar", stacked=True, figsize=(12,6))
    ax.set_title("City stance distribution by state (top 15 states by volume)")
    ax.set_ylabel("Proportion within state")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(figS, dpi=220)
    plt.close()

# Optional city-only: stance over time (if year exists)
year_stance_table = None
if col_year and "city" in w[col_source].unique():
    city = w[w[col_source] == "city"].copy()
    city[col_year] = pd.to_numeric(city[col_year], errors="coerce")
    city = city.dropna(subset=[col_year])
    year_stance = (
        city.groupby([col_year, col_stance]).size().reset_index(name="n")
    )
    year_stance["pct_within_year"] = year_stance.groupby(col_year)["n"].transform(lambda x: x / x.sum())
    year_stance_table = year_stance

    figY = FIG_DIR / "fig_city_stance_trend_by_year.png"
    pv = year_stance.pivot_table(index=col_year, columns=col_stance, values="pct_within_year", aggfunc="sum").fillna(0.0)
    for st in stance_order:
        if st not in pv.columns:
            pv[st] = 0.0
    pv = pv[stance_order]
    ax = pv.plot(figsize=(10,5))
    ax.set_title("City stance trend over time (proportion by year)")
    ax.set_ylabel("Proportion")
    ax.set_xlabel("Year")
    plt.tight_layout()
    plt.savefig(figY, dpi=220)
    plt.close()

# =========================================================
# TABLE 6: Subtopic mismatch (City pct − Amazon pct) within focus themes
# =========================================================
mismatch_rows = []
for theme in FOCUS_THEMES:
    sub = sub_prev[sub_prev[col_theme] == theme].copy()
    if sub.empty:
        continue
    pv = sub.pivot_table(index=col_subname, columns=col_source, values="pct_within_theme", aggfunc="sum").fillna(0.0)
    for s in ["city", "amazon"]:
        if s not in pv.columns:
            pv[s] = 0.0
    pv["diff_city_minus_amazon"] = pv["city"] - pv["amazon"]
    pv = pv.reset_index()
    pv.insert(0, "dominant_theme", theme)
    mismatch_rows.append(pv)

mismatch_table = pd.concat(mismatch_rows, ignore_index=True) if mismatch_rows else pd.DataFrame()
# also create top city-heavy and amazon-heavy views
top_city_heavy = mismatch_table.sort_values("diff_city_minus_amazon", ascending=False).groupby("dominant_theme").head(10)
top_amazon_heavy = mismatch_table.sort_values("diff_city_minus_amazon", ascending=True).groupby("dominant_theme").head(10)

# =========================================================
# Save ALL tables to Excel
# =========================================================
with pd.ExcelWriter(OUT_TABLES, engine="openpyxl") as writer:
    corpus_summary.to_excel(writer, index=False, sheet_name="table2_corpus_summary")
    theme_prev.to_excel(writer, index=False, sheet_name="theme_prevalence")
    subtopic_summary.to_excel(writer, index=False, sheet_name="table3_subtopic_summary_focus")
    stance_dist.to_excel(writer, index=False, sheet_name="stance_distribution")
    stance_by_sub.to_excel(writer, index=False, sheet_name="stance_by_subtopic")
    sent_by_sub.to_excel(writer, index=False, sheet_name="sentiment_by_subtopic")
    if emo_by_sub is not None:
        emo_by_sub.to_excel(writer, index=False, sheet_name="emotion_by_subtopic")
    resistance.to_excel(writer, index=False, sheet_name="table5_resistance_all")
    top_resistance_overall.to_excel(writer, index=False, sheet_name="table5_top_resistance_overall")
    top_resistance_by_theme.to_excel(writer, index=False, sheet_name="table5_top_resistance_by_theme")
    mismatch_table.to_excel(writer, index=False, sheet_name="table6_mismatch_focus")
    top_city_heavy.to_excel(writer, index=False, sheet_name="table6_top_city_heavy")
    top_amazon_heavy.to_excel(writer, index=False, sheet_name="table6_top_amazon_heavy")
    if state_stance_table is not None:
        state_stance_table.to_excel(writer, index=False, sheet_name="city_state_stance")
    if year_stance_table is not None:
        year_stance_table.to_excel(writer, index=False, sheet_name="city_year_stance")

print("\nDONE ")
print(f"Tables saved: {OUT_TABLES.resolve()}")
print(f"Figures saved in: {FIG_DIR.resolve()}")

print("\nNOTE:")
print("- Figure 1 (pipeline/graphical abstract) is typically a manual diagram; the script generates all quantitative figures.")


DONE 
Tables saved: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\spc_results_tables.xlsx
Figures saved in: C:\Users\ermad\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\spc_figures

NOTE:
- Figure 1 (pipeline/graphical abstract) is typically a manual diagram; the script generates all quantitative figures.


In [7]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
INPUT_XLSX = Path("Final results.xlsx")

FOCUS_THEMES = [
    "policy_regulation",
    "packaging_single_use",
    "consumer_tradeoffs"
]

TOP_SUBTOPICS_PER_THEME = 8

OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figure_62"

OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

# =========================================================
# Helper functions
# =========================================================
def norm_header(h):
    return re.sub(r"[^a-z0-9]+", "", str(h).lower())

def find_col(df, candidates):
    cols = list(df.columns)
    m = {norm_header(c): c for c in cols}
    for cand in candidates:
        k = norm_header(cand)
        if k in m:
            return m[k]
    return None

def safe_filename(s):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s))

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_excel(INPUT_XLSX)

# =========================================================
# FIND REQUIRED COLUMNS
# =========================================================
col_source = find_col(df, ["source"])
col_theme = find_col(df, ["dominant_theme", "theme"])
col_subid = find_col(df, ["subtopic_id"])
col_subname = find_col(df, ["subtopic_name", "Subtopic Name"])
col_sent = find_col(df, ["sentiment_compound", "sentiment"])

# =========================================================
# CLEAN DATA
# =========================================================
df[col_source] = df[col_source].astype(str).str.lower().str.strip()
df[col_theme] = df[col_theme].astype(str).str.lower().str.strip()

df[col_sent] = pd.to_numeric(df[col_sent], errors="coerce")
df[col_subid] = pd.to_numeric(df[col_subid], errors="coerce")

df = df.dropna(subset=[col_subid, col_sent])

# =========================================================
# COMPUTE MEAN SENTIMENT BY SUBTOPIC
# =========================================================
sent_by_sub = (
    df.groupby([col_source, col_theme, col_subid, col_subname])[col_sent]
    .agg(mean="mean", n="count")
    .reset_index()
)

# =========================================================
# FIGURE 6 HEATMAP
# =========================================================
for theme in FOCUS_THEMES:

    sub = sent_by_sub[sent_by_sub[col_theme] == theme].copy()

    if sub.empty:
        continue

    # select top subtopics
    totals = (
        sub.groupby(col_subname)["n"]
        .sum()
        .reset_index()
        .sort_values("n", ascending=False)
    )

    top_names = totals.head(TOP_SUBTOPICS_PER_THEME)[col_subname].tolist()

    sub = sub[sub[col_subname].isin(top_names)]

    pv = sub.pivot_table(
        index=col_subname,
        columns=col_source,
        values="mean",
        aggfunc="mean"
    ).fillna(0)

    if pv.empty:
        continue

    cols = list(pv.columns)

    fig_path = FIG_DIR / f"fig6_sentiment_heatmap_{safe_filename(theme)}.png"

    plt.figure(figsize=(8, max(4, 0.5 * len(pv.index) + 2)))

    im = plt.imshow(
        pv.values,
        aspect="auto",
        cmap="viridis",      # blue → green → yellow
        vmin=-1,
        vmax=1
    )

    plt.yticks(range(len(pv.index)), pv.index.tolist())
    plt.xticks(range(len(cols)), cols)

    plt.colorbar(im, label="Mean sentiment (compound)")

    plt.title(f"Mean sentiment by subtopic\nTheme: {theme}")

    # add numeric values
    for i in range(pv.shape[0]):
        for j in range(pv.shape[1]):
            plt.text(
                j,
                i,
                f"{pv.iloc[i, j]:.2f}",
                ha="center",
                va="center",
                color="black"
            )

    plt.tight_layout()

    plt.savefig(fig_path, dpi=300)

    plt.close()

print("Figure 6 heatmaps saved in:", FIG_DIR)

Figure 6 heatmaps saved in: outputs\figure_62


In [9]:
import re
from pathlib import Path
from textwrap import fill

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# CONFIG
# =========================================================
INPUT_XLSX = Path("Final results.xlsx")
SHEET_PREFERENCE = ["Final Results", "results ", "results"]

FOCUS_THEMES = [
    "policy_regulation",
    "packaging_single_use",
    "consumer_tradeoffs"
]
TOP_SUBTOPICS_PER_THEME = 8

OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figure_64"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================
def norm_header(h):
    return re.sub(r"[^a-z0-9]+", "", str(h).strip().lower())

def find_col(df, candidates):
    cols = list(df.columns)
    m = {norm_header(c): c for c in cols}
    for cand in candidates:
        k = norm_header(cand)
        if k in m:
            return m[k]
    return None

def safe_filename(s):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s))[:120]

def wrap_labels(labels, width=26):
    return [fill(str(x), width=width) for x in labels]

# =========================================================
# LOAD SHEET
# =========================================================
if not INPUT_XLSX.exists():
    raise FileNotFoundError(f"Missing file: {INPUT_XLSX.resolve()}")

xls = pd.ExcelFile(INPUT_XLSX, engine="openpyxl")
sheet = None
for s in SHEET_PREFERENCE:
    if s in xls.sheet_names:
        sheet = s
        break
if sheet is None:
    sheet = xls.sheet_names[0]

df = pd.read_excel(INPUT_XLSX, sheet_name=sheet, engine="openpyxl")

# =========================================================
# REQUIRED COLUMNS
# =========================================================
col_source = find_col(df, ["source"])
col_theme = find_col(df, ["dominant_theme", "theme"])
col_subid = find_col(df, ["subtopic_id", "subtopicid"])
col_subname = find_col(df, ["Subtopic Name", "subtopic_name", "topic_name", "subtopicname"])
col_sent = find_col(df, ["sentiment_compound", "sentiment", "sentimentcompound"])

missing = []
for name, col in [
    ("source", col_source),
    ("dominant_theme", col_theme),
    ("subtopic_id", col_subid),
    ("subtopic_name", col_subname),
    ("sentiment_compound", col_sent),
]:
    if col is None:
        missing.append(name)

if missing:
    raise ValueError(
        f"Missing required columns in sheet '{sheet}': {missing}\n"
        f"Available columns:\n{list(df.columns)}"
    )

# =========================================================
# CLEAN DATA
# =========================================================
df[col_source] = df[col_source].astype(str).str.lower().str.strip()
df[col_theme] = df[col_theme].astype(str).str.lower().str.strip()
df[col_subname] = df[col_subname].astype(str).str.strip()
df[col_subid] = pd.to_numeric(df[col_subid], errors="coerce")
df[col_sent] = pd.to_numeric(df[col_sent], errors="coerce")

w = df.dropna(subset=[col_subid, col_sent]).copy()

if w.empty:
    raise ValueError("No usable rows found after filtering for subtopic_id and sentiment.")

# =========================================================
# SENTIMENT BY SUBTOPIC
# =========================================================
sent_by_sub = (
    w.groupby([col_source, col_theme, col_subid, col_subname])[col_sent]
     .agg(mean="mean", median="median", n="count")
     .reset_index()
)

# =========================================================
# STYLE
# =========================================================
sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.1
)

# =========================================================
# FIGURE 6
# =========================================================
for theme in FOCUS_THEMES:
    sub = sent_by_sub[sent_by_sub[col_theme] == theme].copy()
    if sub.empty:
        continue

    totals = (
        sub.groupby(col_subname)["n"]
        .sum()
        .reset_index()
        .sort_values("n", ascending=False)
    )
    top_names = totals.head(TOP_SUBTOPICS_PER_THEME)[col_subname].tolist()
    sub = sub[sub[col_subname].isin(top_names)]

    pv = (
        sub.pivot_table(
            index=col_subname,
            columns=col_source,
            values="mean",
            aggfunc="mean"
        )
        .fillna(0.0)
    )

    if pv.empty:
        continue

    desired_cols = [c for c in ["city", "amazon"] if c in pv.columns]
    other_cols = [c for c in pv.columns if c not in desired_cols]
    pv = pv[desired_cols + other_cols]

    if "city" in pv.columns:
        pv = pv.sort_values("city", ascending=False)

    pv.index = wrap_labels(pv.index.tolist(), width=28)

    fig_w = max(6.5, 1.8 * len(pv.columns) + 3.8)
    fig_h = max(4.8, 0.75 * len(pv.index) + 1.8)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    hm = sns.heatmap(
        pv,
        ax=ax,
        cmap="viridis",          # blue -> green -> yellow
        vmin=-1,
        vmax=1,
        center=0,
        annot=True,
        fmt=".2f",
        linewidths=0.8,
        linecolor="white",
        cbar_kws={
            "label": "Mean sentiment (compound)",
            "shrink": 0.9,
            "pad": 0.02
        },
        annot_kws={
            "fontsize": 9
        }
    )

    pretty_theme = theme.replace("_", " ").title()

    ax.set_title(
        f"Figure 6. Mean sentiment by subtopic\n{pretty_theme}",
        fontsize=13,
        weight="bold",
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.tick_params(axis="x", labelrotation=0, labelsize=10)
    ax.tick_params(axis="y", labelrotation=0, labelsize=10)

    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label("Mean sentiment (compound)", fontsize=10)

    plt.subplots_adjust(left=0.34, right=0.92, top=0.88, bottom=0.10)

    out_png = FIG_DIR / f"fig6_sentiment_heatmap_{safe_filename(theme)}.png"
    out_pdf = FIG_DIR / f"fig6_sentiment_heatmap_{safe_filename(theme)}.pdf"

    plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.savefig(out_pdf, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    print(f"Saved: {out_png.resolve()}")
    print(f"Saved: {out_pdf.resolve()}")

print("\nDone.")
print(f"Figure 6 outputs saved in: {FIG_DIR.resolve()}")

Saved: C:\Users\mponnusa\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\figure_64\fig6_sentiment_heatmap_policy_regulation.png
Saved: C:\Users\mponnusa\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\figure_64\fig6_sentiment_heatmap_policy_regulation.pdf
Saved: C:\Users\mponnusa\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\figure_64\fig6_sentiment_heatmap_packaging_single_use.png
Saved: C:\Users\mponnusa\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\figure_64\fig6_sentiment_heatmap_packaging_single_use.pdf
Saved: C:\Users\mponnusa\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\figure_64\fig6_sentiment_heatmap_consumer_tradeoffs.png
Saved: C:\Users\mponnusa\OneDrive - purdue.edu\Papers by Madhu\Sustainable Public opinion\outputs\figure_64\fig6_sentiment_heatmap_consumer_tradeoffs.pdf

Done.
Figure 6 outputs saved in: C:\Users\mponnusa\OneDrive - purdue.edu\